In [ ]:
import os

if not os.getcwd().endswith("root-obj-perf"):
    os.chdir("root-obj-perf")
print(f"Changed working directory to: {os.getcwd()}")


import json
import uproot
import awkward as ak
import vector
import numpy as np
import matplotlib.pyplot as plt
import warnings

# plotting params
plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "axes.grid": True,
        "grid.alpha": 0.6,
        "grid.linestyle": "--",
        "font.size": 14,
        "figure.dpi": 200,
    }
)

# Suppress a harmless warning from the vector library with awkward arrays
warnings.filterwarnings("ignore", message="Passing an awkward array to a ufunc")

# Register the vector library with awkward array
ak.behavior.update(vector.backends.awkward.behavior)

# --- CONFIGURATION ---
with open("hh-bbbb-obj-config.json", "r") as config_file:
    config = json.load(config_file)

with open("config_part.json", "r") as config_file:
    CONFIG_PART = json.load(config_file)

## Load and test model

In [ ]:
import os
import json
import torch
import wandb
from parT import ParticleTransformer
from parT_helpers import extract_wandb_run_id

from train_part_data import L1JetDataset, stratified_split, CombinedJetDataLoader
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


with open("config_part.json", "r") as f:
    config_part = json.load(f)

if not os.getcwd().endswith("root-obj-perf"):
    os.chdir("root-obj-perf")
print(os.getcwd())


device = "mps"

# run_path = "adityatandon29/L1-BTagging-ParT/4w2yuysk"
# run_id = extract_wandb_run_id(run_path)
# model_ckpt = torch.load(
#     "/Users/adityatandon/Documents/College/Physics/Year 4/Neural SBI/root-obj-perf/final_model.pth",
#     weights_only=True,
#     map_location=device,
# )
# model.load_state_dict(model_ckpt)


wandb.init(project="part-btag-analysis")
artifact_path = f"{config_part['wandb']['entity']}/{config_part['wandb']['project']}/{config_part['wandb']['artifact_name']}:{config_part['wandb']['ckpt_type']}"
artifact = wandb.use_artifact(
    artifact_path,
    type="model",
)
artifact_dir = artifact.download()
wandb.finish()
print(f"Model artifact downloaded from W&B: {artifact_dir}")
if os.path.exists(artifact_dir):
    print("Loading checkpoint")
    checkpoint_dir = os.path.join(artifact_dir, os.listdir(artifact_dir)[0])
    checkpoint = torch.load(checkpoint_dir, map_location=device, weights_only=False)
    config_part = checkpoint["config"]

    print(f"Exp name: {config_part['exp_name']}")
    try:
        print(f"Data path: {config_part['data_path']}")
        torch.manual_seed(42)

        dataset = L1JetDataset(config_part["data_path"])
        num_classes = config_part["model"]["num_classes"]

        # Perform stratified split
        train_dataset, val_dataset, train_indices, val_indices, stratify_labels = (
            stratified_split(
                dataset=dataset,
                val_split=config_part["training"]["val_split"],
                num_classes=num_classes,
                random_state=42,
                verbose=True,
            )
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=config_part["training"]["batch_size"],
            shuffle=True,
            num_workers=config_part["training"]["num_workers"],
            pin_memory=True,
            persistent_workers=True,
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=config_part["training"]["batch_size"],
            shuffle=False,
            num_workers=config_part["training"]["num_workers"],
            pin_memory=True,
            persistent_workers=True,
        )
        print("Data loaders prepared.")

    except KeyError:
        print("\nData path not found in checkpoint config")
        print("Using the new config structure.")
        dataset_used = config_part["training"]["data"]["use_dataset"]
        config_part["data_path"] = config_part["training"]["data"][
            f"{dataset_used}_data_path"
        ]
        config_part["training"]["val_split"] = config_part["training"]["data"][
            "val_split"
        ]
        config_part["training"]["val_split"] = config_part["training"]["data"][
            "val_split"
        ]
        config_part["training"]["num_workers"] = config_part["training"]["data"][
            "num_workers"
        ]
        print(f"Data path: {config_part['data_path']}")
        pt_regression = config_part.get("model", {}).get("pt_regression", False)
        combined_loader = CombinedJetDataLoader(
            pf_data_path=config_part["training"]["data"]["pf_data_path"],
            puppi_data_path=config_part["training"]["data"]["puppi_data_path"],
            val_split=config_part["training"]["data"]["val_split"],
            batch_size=config_part["training"]["batch_size"],
            match_mode=config_part["training"]["data"]["match_mode"],
            num_workers=2,
            random_state=42,
            # n_bins_pt=20,
            # n_bins_eta=20,
            # reweight_max_pt=500.0,
            # reweight_clip_max=100,
            # verbose=True,
            # smooth_sigma=0.0,
            # pt_min=40.0,
            # pt_max=70.0,
            pt_regression=pt_regression,
            # eta_min=-1.0,
            # eta_max=1.0,
        )
        if config_part["training"]["data"]["use_dataset"] == "pf":
            print("\nUsing PF dataset for training.")
            (
                train_loader,
                train_indices,
                val_loader,
                val_indices,
                train_labels,
                val_labels,
            ) = combined_loader.get_pf_loaders(shuffle=True)
        elif config_part["training"]["data"]["use_dataset"] == "puppi":
            print("\nUsing PUPPI dataset for training.")
            (
                train_loader,
                train_indices,
                val_loader,
                val_indices,
                train_labels,
                val_labels,
            ) = combined_loader.get_puppi_loaders(shuffle=True)
        print(
            f"Data loaders prepared with {len(train_loader.dataset)} training samples and {len(val_loader.dataset)} validation samples."
        )

    print(f"\nEpoch: {checkpoint['epoch']}")
    print(f"Val loss: {checkpoint['val_loss']}")
    print(f"Val_auc: {checkpoint['val_auc']}")
    pt_regression = config_part.get("model", {}).get("pt_regression", False)
    quantile_regression = config_part.get("model", {}).get("quantile_regression", False)
    model = ParticleTransformer(
        input_dim=config_part["input_dim"],
        embed_dim=config_part["model"]["embed_dim"],
        num_heads=config_part["model"]["num_heads"],
        num_layers=config_part["model"]["num_layers"],
        num_cls_layers=config_part["model"]["num_cls_layers"],
        dropout=config_part["model"]["dropout"],
        num_classes=config_part["model"]["num_classes"],
        pt_regression=pt_regression,
        quantile_regression=quantile_regression,
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(
        f"Checkpoint loaded (pt_regression={pt_regression}, quantile_regression={quantile_regression})"
    )

# Store as module-level for later cells
# CONFIG_PART = config_part

In [ ]:
# Run inference on validation set
model.eval()
all_outputs = []
all_labels = []
all_constituents = []  # Store full constituent data for jet reconstruction
all_reg_pt = []  # Regression head outputs (if pt_regression)
all_quantiles = []  # Quantile regression head outputs (if quantile_regression)
all_jet_pt_val = []  # Reco jet pT from dataset
all_gen_pt_val = []  # Gen pT from dataset

with torch.no_grad():
    for X_batch, y_batch, mask_batch, _, jet_pt_batch, _, gen_pt_batch in tqdm(
        val_loader
    ):
        all_constituents.append(X_batch.cpu())
        X_batch = X_batch.to(device)
        mask_batch = mask_batch.to(device)
        y_batch = y_batch.to(device)

        out = model(X_batch, particle_mask=mask_batch)
        cls_logits = out["classification"]
        outputs = torch.nn.functional.sigmoid(cls_logits).squeeze()

        all_outputs.append(outputs.cpu())
        all_labels.append(y_batch.cpu())
        all_jet_pt_val.append(jet_pt_batch.squeeze().cpu())
        all_gen_pt_val.append(gen_pt_batch.squeeze().cpu())

        if "pt" in out:
            all_reg_pt.append(out["pt"].squeeze().cpu())
        if "quantiles" in out:
            all_quantiles.append(out["quantiles"].cpu())

all_outputs = torch.cat(all_outputs).numpy()
all_labels = torch.cat(all_labels).numpy()
all_constituents = torch.cat(all_constituents)
all_jet_pt_val = torch.cat(all_jet_pt_val).numpy()
all_gen_pt_val = torch.cat(all_gen_pt_val).numpy()
has_regression = len(all_reg_pt) > 0
if has_regression:
    all_reg_pt = torch.cat(all_reg_pt).numpy()
    print(f"Regression outputs collected: {all_reg_pt.shape}")

has_quantile = len(all_quantiles) > 0
if has_quantile:
    all_quantiles = torch.cat(all_quantiles).numpy()
    print(f"Quantile regression outputs collected: {all_quantiles.shape}")
    print(
        f"  q16 range: {all_quantiles[:, 0].min():.4f} – {all_quantiles[:, 0].max():.4f}"
    )
    print(
        f"  q84 range: {all_quantiles[:, 1].min():.4f} – {all_quantiles[:, 1].max():.4f}"
    )
else:
    print("No quantile regression head in this model.")


# Reconstruct jet 4-vectors using vector library (proper 4-vector addition)
# Constituent format: [mass, pt, eta, phi, ...]
const_mass = all_constituents[:, :, 0].numpy()
const_pt = all_constituents[:, :, 1].numpy()
const_eta = all_constituents[:, :, 2].numpy()
const_phi = all_constituents[:, :, 3].numpy()

# Create constituent 4-vectors and sum to get jet 4-vectors
const_vectors = vector.array(
    {
        "pt": const_pt,
        "eta": const_eta,
        "phi": const_phi,
        "mass": const_mass,
    }
)
# Sum over constituents (axis=1) to get jet 4-vector
jet_vectors = const_vectors.sum(axis=1)

# Extract jet kinematics
val_jet_pt = jet_vectors.pt
val_jet_eta = jet_vectors.eta
val_jet_phi = jet_vectors.phi
val_jet_mass = jet_vectors.mass

print(f"\nJet kinematics reconstructed using vector library:")
print(f"  pT range: {val_jet_pt.min():.2f} - {val_jet_pt.max():.2f} GeV")
print(f"  eta range: {val_jet_eta.min():.2f} - {val_jet_eta.max():.2f}")

# Print regression info if available
if has_regression:
    sig_mask = all_labels.squeeze() == 1
    print(f"\nRegression summary (signal jets only):")
    print(
        f"  Predicted correction range: {all_reg_pt[sig_mask].min():.3f} – {all_reg_pt[sig_mask].max():.3f}"
    )
    print(
        f"  True correction (gen_pt/jet_pt) range: {(all_gen_pt_val[sig_mask] / (all_jet_pt_val[sig_mask] + 1e-9)).min():.3f} – {(all_gen_pt_val[sig_mask] / (all_jet_pt_val[sig_mask] + 1e-9)).max():.3f}"
    )

In [ ]:
print(f"Total validation samples: {len(all_outputs)}")
print(f"Num positive samples: {np.sum(all_labels)}")
print(f"Num negative samples: {len(all_outputs) - np.sum(all_labels)}")
print(f"% of positive samples: {100 * np.sum(all_labels) / len(all_outputs):.2f}%")

In [ ]:
# ============================================================
# REGRESSION HEAD ANALYSIS
# Uses model, val_loader, and regression outputs from cells above.
# ============================================================
signal_mask = all_labels.squeeze() == 1
jet_pt = all_jet_pt_val
gen_pt = all_gen_pt_val

print(f"Total validation jets: {len(all_labels)}")
print(f"  Signal jets: {signal_mask.sum()}")
print(f"  Background jets: {(~signal_mask).sum()}")
if has_regression:
    reg_pt = all_reg_pt
    print(f"\nRegression output (signal only):")
    print(
        f"  reg_pt  range: {reg_pt[signal_mask].min():.2f} – {reg_pt[signal_mask].max():.2f}"
    )
    print(
        f"  jet_pt  range: {jet_pt[signal_mask].min():.2f} – {jet_pt[signal_mask].max():.2f}"
    )
    print(
        f"  gen_pt  range: {gen_pt[signal_mask].min():.2f} – {gen_pt[signal_mask].max():.2f}"
    )
    correction = gen_pt[signal_mask] / (jet_pt[signal_mask] + 1e-9)
    print(
        f"  Correction factor (gen_pt / jet_pt) range: {correction.min():.3f} – {correction.max():.3f}"
    )
else:
    print("\nNo regression head in this model.")

# --- Plots ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. jet_pt distribution (signal vs background)
ax = axes[0, 0]
ax.hist(
    jet_pt[signal_mask],
    bins=60,
    range=(0, 500),
    histtype="step",
    density=True,
    label="Signal",
)
ax.hist(
    jet_pt[~signal_mask],
    bins=60,
    range=(0, 500),
    histtype="step",
    density=True,
    label="Background",
)
ax.set_xlabel("Reco jet $p_T$ [GeV]")
ax.set_ylabel("Density")
ax.set_title("Reco jet $p_T$ distribution")
ax.legend()

# 2. gen_pt distribution (signal only — bkg has gen_pt=0)
ax = axes[0, 1]
ax.hist(
    gen_pt[signal_mask],
    bins=60,
    range=(0, 500),
    histtype="step",
    density=True,
    color="C0",
)
ax.set_xlabel("Gen $p_T$ [GeV]")
ax.set_ylabel("Density")
ax.set_title("Gen $p_T$ (signal jets only)")

# 3. Correction factor = gen_pt / jet_pt (signal only)
ax = axes[0, 2]
correction = gen_pt[signal_mask] / (jet_pt[signal_mask] + 1e-9)
ax.hist(correction, bins=60, range=(0, 3), histtype="step", density=True, color="C2")
ax.axvline(1.0, color="gray", linestyle="--", alpha=0.7, label="No correction")
ax.set_xlabel("gen $p_T$ / reco $p_T$")
ax.set_ylabel("Density")
ax.set_title("Target correction factor (signal)")
ax.legend()

if has_regression:
    # 4. Regression output distribution
    ax = axes[1, 0]
    ax.hist(reg_pt[signal_mask], bins=60, histtype="step", density=True, label="Signal")
    ax.hist(
        reg_pt[~signal_mask], bins=60, histtype="step", density=True, label="Background"
    )
    ax.set_xlabel("Regression output (predicted correction)")
    ax.set_ylabel("Density")
    ax.set_title("Regression head output")
    ax.legend()

    # 5. Predicted correction vs true correction (signal only, scatter)
    ax = axes[1, 1]
    true_corr = gen_pt[signal_mask] / (jet_pt[signal_mask] + 1e-9)
    pred_corr = reg_pt[signal_mask]
    ax.scatter(true_corr, pred_corr, s=1, alpha=0.3)
    lim = max(true_corr.max(), pred_corr.max()) * 1.05
    ax.plot([0, lim], [0, lim], "r--", alpha=0.5, label="Ideal")
    ax.set_xlim(0, lim)
    ax.set_xlabel("True correction (gen $p_T$ / reco $p_T$)")
    ax.set_ylabel("Predicted correction")
    ax.set_title("Predicted vs true correction (signal)")
    ax.legend()

    # 6. Corrected jet pT vs gen pT (signal)
    ax = axes[1, 2]
    corrected_pt = jet_pt[signal_mask] * pred_corr
    ax.scatter(gen_pt[signal_mask], corrected_pt, s=1, alpha=0.3, label="Corrected")
    ax.scatter(
        gen_pt[signal_mask],
        jet_pt[signal_mask],
        s=1,
        alpha=0.1,
        color="gray",
        label="Uncorrected",
    )
    lim = 500
    ax.plot([0, lim], [0, lim], "r--", alpha=0.5)
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_xlabel("Gen $p_T$ [GeV]")
    ax.set_ylabel("Jet $p_T$ [GeV]")
    ax.set_title("Corrected vs uncorrected $p_T$ (signal)")
    ax.legend()
else:
    for i in range(3):
        axes[1, i].text(
            0.5,
            0.5,
            "No regression head",
            ha="center",
            va="center",
            transform=axes[1, i].transAxes,
        )
        axes[1, i].set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# training data
all_constituents_train = []
all_labels_train = []
all_masks_train = []
all_weights_train = []

for X_batch, y_batch, mask_batch, weights_batch, _, _, _ in tqdm(train_loader):
    all_constituents_train.append(X_batch.cpu())
    all_labels_train.append(y_batch.squeeze(-1).cpu())
    all_masks_train.append(mask_batch.cpu())
    all_weights_train.append(weights_batch.cpu())

all_constituents_train = torch.concat(all_constituents_train)
all_labels_train = torch.concat(all_labels_train)
all_masks_train = torch.concat(all_masks_train)
all_weights_train = torch.concat(all_weights_train)

const_mass_train = all_constituents_train[:, :, 0].numpy()
const_pt_train = all_constituents_train[:, :, 1].numpy()
const_eta_train = all_constituents_train[:, :, 2].numpy()
const_phi_train = all_constituents_train[:, :, 3].numpy()

# Create constituent 4-vectors and sum to get jet 4-vectors
const_vectors_train = vector.array(
    {
        "pt": const_pt_train,
        "eta": const_eta_train,
        "phi": const_phi_train,
        "mass": const_mass_train,
    }
)
# Sum over constituents (axis=1) to get jet 4-vector
jet_vectors_train = const_vectors_train.sum(axis=1)

# Extract jet kinematics
train_jet_pt = jet_vectors_train.pt
train_jet_eta = jet_vectors_train.eta
train_jet_phi = jet_vectors_train.phi
train_jet_mass = jet_vectors_train.mass

In [ ]:
# train signal/background vs val signal/background plots
fig, ax = plt.subplots()
ax.hist(
    train_jet_pt[all_labels_train == 1],
    bins=50,
    range=(0, 500),
    density=True,
    alpha=0.5,
    label="Train Signal",
)
ax.hist(
    train_jet_pt[all_labels_train == 0],
    bins=50,
    range=(0, 500),
    density=True,
    alpha=0.5,
    label="Train Background",
)
ax.hist(
    val_jet_pt[all_labels.reshape(all_labels.shape[0]) == 1],
    bins=50,
    range=(0, 500),
    density=True,
    histtype="step",
    label="Val Signal",
)
ax.hist(
    val_jet_pt[all_labels.reshape(all_labels.shape[0]) == 0],
    bins=50,
    range=(0, 500),
    density=True,
    histtype="step",
    label="Val Background",
)
ax.set_xlabel("Jet $p_T$ [GeV]")
ax.set_ylabel("Normalized Entries")
ax.set_title("Jet $p_T$ Distribution: Train vs Val")
ax.legend()
plt.show()

# reweighted training data plot
fig, ax = plt.subplots()
ax.hist(
    train_jet_pt[all_labels_train == 1],
    bins=50,
    range=(0, 500),
    weights=all_weights_train[all_labels_train == 1],
    density=True,
    alpha=0.5,
    label="Reweighted Train Signal",
)
ax.hist(
    train_jet_pt[all_labels_train == 0],
    bins=50,
    range=(0, 500),
    weights=all_weights_train[all_labels_train == 0],
    density=True,
    alpha=0.5,
    label="Reweighted Train Background",
)
ax.set_xlabel("Jet $p_T$ [GeV]")
ax.set_ylabel("Normalized Entries")
ax.set_title("Reweighted Jet $p_T$ Distribution: Train Set")
ax.legend()
plt.show()


fig, ax = plt.subplots()
ax.hist(
    train_jet_eta[all_labels_train == 1],
    bins=50,
    range=(-3, 3),
    weights=all_weights_train[all_labels_train == 1],
    density=True,
    alpha=0.5,
    label="Reweighted Train Signal",
)
ax.hist(
    train_jet_eta[all_labels_train == 0],
    bins=50,
    range=(-3, 3),
    weights=all_weights_train[all_labels_train == 0],
    density=True,
    alpha=0.5,
    label="Reweighted Train Background",
)
ax.set_xlabel("Jet $\\eta$ [GeV]")
ax.set_ylabel("Normalized Entries")
ax.set_title("Reweighted Jet $\\eta$ Distribution: Train Set")
ax.legend()
plt.show()

n_const_train = all_masks_train.sum(dim=1).cpu().numpy()
n_const_val = (all_constituents[:, :, 1] > 0).sum(dim=1).cpu().numpy()

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Training data
h0 = ax[0].hist2d(train_jet_eta, n_const_train, bins=50, cmap="viridis", cmin=1)
ax[0].set_xlabel(r"Jet $\eta$")
ax[0].set_ylabel("Number of Constituents")
ax[0].set_title("Train Set: $N_{const}$ vs $\eta$")
plt.colorbar(h0[3], ax=ax[0], label="Counts")

# Validation data
h1 = ax[1].hist2d(val_jet_eta, n_const_val, bins=50, cmap="viridis", cmin=1)
ax[1].set_xlabel(r"Jet $\eta$")
ax[1].set_ylabel("Number of Constituents")
ax[1].set_title("Validation Set: $N_{const}$ vs $\eta$")
plt.colorbar(h1[3], ax=ax[1], label="Counts")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ROC ANALYSIS AND COMPARISON WITH OTHER TAGGERS
# ============================================================
import copy
from data_loading_helpers import (
    apply_custom_cuts,
    load_and_prepare_data,
    select_gen_b_quarks_from_higgs,
)
from analysis_helpers import get_purity_mask_cross_matched, calculate_roc_points
from plotting_helpers import plot_roc_comparison
from sklearn.metrics import roc_curve, auc

with open("hh-bbbb-obj-config.json", "r") as f:
    config = json.load(f)

# Create vector array from reconstructed jet kinematics for cuts
val_jet_vectors = vector.array(
    {
        "pt": val_jet_pt,
        "eta": val_jet_eta,
        "phi": val_jet_phi,
        "mass": val_jet_mass,
    }
)

# Apply kinematic cuts
val_cuts_mask = apply_custom_cuts(
    val_jet_vectors, config, "l1barrelextpf", kinematic_only=True, return_jets=False
)
print(f"Jets after kinematic cuts: {np.sum(val_cuts_mask)}/{len(val_cuts_mask)}")

all_labels = all_labels.reshape(-1)
all_labels_after_cuts = all_labels[val_cuts_mask]
all_outputs_after_cuts = all_outputs[val_cuts_mask]

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(all_labels_after_cuts, all_outputs_after_cuts)
roc_auc = auc(fpr, tpr)

# Load reference collections for comparison
events = load_and_prepare_data(
    config["file_pattern"],
    config["tree_name"],
    [
        "GenPart",
        config["offline"]["collection_name"],
        config["l1ng"]["collection_name"],
        config["l1ext"]["collection_name"],
    ],
    config["max_events"],
    correct_pt=True,
)

# UParT config
upart_CONFIG = copy.deepcopy(config)
upart_CONFIG["offline"]["tagger_name"] = "btagUParTAK4B"
upart_CONFIG["offline"]["b_tag_cut"] = 0.00496

upart_events = load_and_prepare_data(
    upart_CONFIG["file_pattern"],
    upart_CONFIG["tree_name"],
    [upart_CONFIG["offline"]["collection_name"]],
    upart_CONFIG["max_events"],
    CONFIG=upart_CONFIG,
)

# Gen-level b-quarks for matching
gen_b_quarks = select_gen_b_quarks_from_higgs(events)
gen_b_quarks = gen_b_quarks[
    (gen_b_quarks.pt > config["gen"]["pt_cut"])
    & (abs(gen_b_quarks.eta) < config["gen"]["eta_cut"])
]

# Apply cuts and compute ROC for each collection
offline_roc_events = apply_custom_cuts(
    events[config["offline"]["collection_name"]], config, "offline", kinematic_only=True
)
offline_upart_roc_events = apply_custom_cuts(
    upart_events[upart_CONFIG["offline"]["collection_name"]],
    upart_CONFIG,
    "offline",
    kinematic_only=True,
)
l1ng_roc_events = apply_custom_cuts(
    events[config["l1ng"]["collection_name"]], config, "l1ng", kinematic_only=True
)
l1ext_roc_events = apply_custom_cuts(
    events[config["l1ext"]["collection_name"]], config, "l1ext", kinematic_only=True
)

offline_roc_mask = get_purity_mask_cross_matched(gen_b_quarks, offline_roc_events)
offline_roc_mask_upart = get_purity_mask_cross_matched(
    gen_b_quarks, offline_upart_roc_events
)
l1ng_roc_mask = get_purity_mask_cross_matched(gen_b_quarks, l1ng_roc_events)
l1ext_roc_mask = get_purity_mask_cross_matched(gen_b_quarks, l1ext_roc_events)

offline_roc = calculate_roc_points(
    offline_roc_events, offline_roc_mask, config["offline"]["tagger_name"]
)
offline_roc_upart = calculate_roc_points(
    offline_upart_roc_events,
    offline_roc_mask_upart,
    upart_CONFIG["offline"]["tagger_name"],
)
l1ng_roc = calculate_roc_points(
    l1ng_roc_events, l1ng_roc_mask, config["l1ng"]["tagger_name"]
)
l1ext_roc = calculate_roc_points(
    l1ext_roc_events, l1ext_roc_mask, config["l1ext"]["tagger_name"]
)


# Print working point summaries
def get_roc_point_at_mistag(mistag, eff, thresh, target_mistag):
    thresh = np.sort(thresh)
    return [(m, e, th) for m, e, th in zip(mistag, eff, thresh) if m >= target_mistag][
        0
    ]


def get_working_points(name, roc_data):
    wps = []
    mistag, eff, auc_val, thresh = roc_data
    print(f"\n{name} (AUC: {auc_val:.5f})")
    for wp_name, target in [("Tight", 0.001), ("Medium", 0.01), ("Loose", 0.1)]:
        fpr_wp, tpr_wp, thresh_wp = get_roc_point_at_mistag(mistag, eff, thresh, target)
        print(
            f"  {wp_name}: TPR={tpr_wp*100:.2f}%, 1/FPR={1/fpr_wp:.1f}, thresh={thresh_wp:.4f}"
        )
        wps.append(thresh_wp)
    return wps


pnet_wps = get_working_points("Offline PNet", offline_roc)
upart_wps = get_working_points("Offline UParT", offline_roc_upart)
l1ng_wps = get_working_points("L1NG", l1ng_roc)
l1ext_wps = get_working_points("L1ExtJet", l1ext_roc)
part_wps = get_working_points("Trained ParT", (fpr, tpr, roc_auc, thresholds))

In [ ]:
# ============================================================
# PR CURVE + S/sqrt(B) VS THRESHOLD + PER-BIN EFFICIENCY TABLES
# ============================================================
import pandas as pd
from sklearn.metrics import precision_recall_curve, average_precision_score

# Use post-cuts arrays if available
labels = all_labels_after_cuts
scores = all_outputs_after_cuts
jet_pt = val_jet_pt[val_cuts_mask]
jet_eta = val_jet_eta[val_cuts_mask]

# Plot output directory for new plots
run_id = config_part.get("exp_name", "unknown_run").replace("/", "_")
artifact_name = CONFIG_PART.get("wandb", {}).get("artifact_name", "unknown_artifact")
artifact_type = CONFIG_PART.get("wandb", {}).get("ckpt_type", "unknown_type")
plot_dir = f"../Updates/plots_{run_id}/{artifact_name}:{artifact_type}"
os.makedirs(plot_dir, exist_ok=True)


def save_fig_local(fig, name):
    filepath = os.path.join(plot_dir, f"{name}.png")
    fig.savefig(filepath, dpi=150, bbox_inches="tight")
    print(f"  Saved: {name}.png")


# ------------------------------
# 1) Precision-Recall curve
# ------------------------------
precision, recall, pr_thresholds = precision_recall_curve(labels, scores)
pr_auc = average_precision_score(labels, scores)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall, precision, color="purple")
ax.set_xlabel("Recall (Signal Efficiency)")
ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall Curve (post-cuts) | PR-AUC={pr_auc:.4f}")
ax.grid(True, alpha=0.3)
save_fig_local(fig, "precision_recall_curve")
plt.show()

# ------------------------------
# 2) S/sqrt(B) vs threshold
# ------------------------------
# Use unique thresholds from scores for stability
thr = np.linspace(0, 1, 200)
s_over_root_b = []
s_counts = []
b_counts = []

for t in thr:
    preds = scores >= t
    s = ((preds == 1) & (labels == 1)).sum()
    b = ((preds == 1) & (labels == 0)).sum()
    s_counts.append(s)
    b_counts.append(b)
    s_over_root_b.append(s / np.sqrt(b + 1e-9))

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(thr, s_over_root_b, color="darkgreen")
ax.set_xlabel("Threshold")
ax.set_ylabel(r"$S/\sqrt{B}$")
ax.set_title(r"$S/\sqrt{B}$ vs Threshold (post-cuts)")
ax.grid(True, alpha=0.3)
save_fig_local(fig, "s_over_root_b_vs_threshold")
plt.show()

# ------------------------------
# 3) Per-bin efficiency tables
# ------------------------------
pt_bins = [25, 100, 200, 300, 400, 500, np.inf]
eta_bins = [0.0, 0.5, 1.0, 1.5, 2.4]
working_points = pnet_wps


def efficiency_table(pt_bins, eta_bins, labels, scores, jet_pt, jet_eta, wp):
    rows = []
    for i in range(len(pt_bins) - 1):
        for j in range(len(eta_bins) - 1):
            pt_lo, pt_hi = pt_bins[i], pt_bins[i + 1]
            eta_lo, eta_hi = eta_bins[j], eta_bins[j + 1]
            bin_mask = (
                (jet_pt >= pt_lo)
                & (jet_pt < pt_hi)
                & (np.abs(jet_eta) >= eta_lo)
                & (np.abs(jet_eta) < eta_hi)
            )
            bin_labels = labels[bin_mask]
            bin_scores = scores[bin_mask]
            n = len(bin_labels)
            if n == 0:
                rows.append(
                    {
                        "pt_bin": f"[{pt_lo},{pt_hi})",
                        "eta_bin": f"[{eta_lo},{eta_hi})",
                        "N": 0,
                        "sig_eff": np.nan,
                        "bkg_eff": np.nan,
                        "bkg_rej": np.nan,
                    }
                )
                continue
            sig_mask = bin_labels == 1
            bkg_mask = bin_labels == 0
            sig_eff = np.mean(bin_scores[sig_mask] >= wp) if sig_mask.any() else np.nan
            bkg_eff = np.mean(bin_scores[bkg_mask] >= wp) if bkg_mask.any() else np.nan
            bkg_rej = 1.0 / bkg_eff if bkg_eff and bkg_eff > 0 else np.inf
            rows.append(
                {
                    "pt_bin": f"[{pt_lo},{pt_hi})",
                    "eta_bin": f"[{eta_lo},{eta_hi})",
                    "N": n,
                    "sig_eff": sig_eff,
                    "bkg_eff": bkg_eff,
                    "bkg_rej": bkg_rej,
                }
            )
    return pd.DataFrame(rows)


for wp in working_points:
    df = efficiency_table(pt_bins, eta_bins, labels, scores, jet_pt, jet_eta, wp)
    print(f"\nPer-bin efficiencies @ threshold = {wp}")
    display(df)


# ------------------------------
# 4) PR-AUC and optimal S/sqrt(B) per kinematic bin
# ------------------------------
def pr_auc_and_opt_s_over_root_b(
    pt_bins, eta_bins, labels, scores, jet_pt, jet_eta, thr_grid
):
    rows = []
    for i in range(len(pt_bins) - 1):
        for j in range(len(eta_bins) - 1):
            pt_lo, pt_hi = pt_bins[i], pt_bins[i + 1]
            eta_lo, eta_hi = eta_bins[j], eta_bins[j + 1]
            bin_mask = (
                (jet_pt >= pt_lo)
                & (jet_pt < pt_hi)
                & (np.abs(jet_eta) >= eta_lo)
                & (np.abs(jet_eta) < eta_hi)
            )
            bin_labels = labels[bin_mask]
            bin_scores = scores[bin_mask]
            n = len(bin_labels)
            if n < 10 or len(np.unique(bin_labels)) < 2:
                rows.append(
                    {
                        "pt_bin": f"[{pt_lo},{pt_hi})",
                        "eta_bin": f"[{eta_lo},{eta_hi})",
                        "N": n,
                        "pr_auc": np.nan,
                        "s_over_root_b_max": np.nan,
                        "thr_opt": np.nan,
                        "s_at_opt": np.nan,
                        "b_at_opt": np.nan,
                    }
                )
                continue
            pr_auc_bin = average_precision_score(bin_labels, bin_scores)
            srb_vals = []
            s_vals = []
            b_vals = []
            for t in thr_grid:
                preds = bin_scores >= t
                s = ((preds == 1) & (bin_labels == 1)).sum()
                b = ((preds == 1) & (bin_labels == 0)).sum()
                srb_vals.append(s / np.sqrt(b + 1e-9))
                s_vals.append(s)
                b_vals.append(b)
            srb_vals = np.array(srb_vals)
            opt_idx = int(np.nanargmax(srb_vals))
            rows.append(
                {
                    "pt_bin": f"[{pt_lo},{pt_hi})",
                    "eta_bin": f"[{eta_lo},{eta_hi})",
                    "N": n,
                    "pr_auc": pr_auc_bin,
                    "s_over_root_b_max": srb_vals[opt_idx],
                    "thr_opt": thr_grid[opt_idx],
                    "s_at_opt": s_vals[opt_idx],
                    "b_at_opt": b_vals[opt_idx],
                }
            )
    return pd.DataFrame(rows)


df_pr_srb = pr_auc_and_opt_s_over_root_b(
    pt_bins, eta_bins, labels, scores, jet_pt, jet_eta, thr
)
print("\nPer-bin PR-AUC and optimal S/sqrt(B)")
display(df_pr_srb)

In [ ]:
# ============================================================
# LOAD FULL DATASET FOR CONSTITUENT ANALYSIS
# ============================================================
from train_part_data import L1JetDataset
from data_loading_helpers import load_and_prepare_data, apply_custom_cuts


full_dataset = L1JetDataset(filepath=config_part["data_path"])
x_full, y_full, mask_full, _ = full_dataset[:]

# Constituent kinematics
const_mass_full = x_full[:, :, 0].numpy()
const_pt_full = x_full[:, :, 1].numpy()
const_eta_full = x_full[:, :, 2].numpy()
const_phi_full = x_full[:, :, 3].numpy()

# Reconstruct jet kinematics using vector library
const_vectors_full = vector.array(
    {
        "pt": const_pt_full,
        "eta": const_eta_full,
        "phi": const_phi_full,
        "mass": const_mass_full,
    }
)
jet_vectors_full = const_vectors_full.sum(axis=1)

# Feature names for constituent plots (excluding particle IDs)
feature_names = [
    "Mass",
    "Pt",
    "Eta",
    "Phi",
    "Dxy",
    "Z0",
    "Charge",
    "Log Pt Rel",
    "Delta Eta",
    "Delta Phi",
    "PUPPI weight",
    "Log Delta R",
]
x_features = x_full[:, :, : len(feature_names)]

# Load reference jet collections for comparison
with open("hh-bbbb-obj-config.json", "r") as f:
    config = json.load(f)

events = load_and_prepare_data(
    config["file_pattern"],
    config["tree_name"],
    [config["offline"]["collection_name"], config["l1ng"]["collection_name"]],
    config["max_events"],
    correct_pt=True,
)

offline_jets = apply_custom_cuts(
    events[config["offline"]["collection_name"]], config, "offline", kinematic_only=True
)
l1ng_jets = apply_custom_cuts(
    events[config["l1ng"]["collection_name"]], config, "l1ng", kinematic_only=True
)

# Flatten for histograms
offline_pt = ak.to_numpy(ak.flatten(offline_jets.pt))
offline_eta = ak.to_numpy(ak.flatten(offline_jets.eta))
l1ng_pt = ak.to_numpy(ak.flatten(l1ng_jets.pt))
l1ng_eta = ak.to_numpy(ak.flatten(l1ng_jets.eta))

print(f"Dataset loaded: {len(x_full)} jets")
print(
    f"Reconstructed jet pT range: {jet_vectors_full.pt.min():.2f} - {jet_vectors_full.pt.max():.2f} GeV"
)
print(
    f"Reconstructed jet eta range: {jet_vectors_full.eta.min():.2f} - {jet_vectors_full.eta.max():.2f}"
)

## Generate and Save All Plots

This cell generates all analysis plots and saves them to `../Updates/plots_{run_id}/`

In [ ]:
run_id = config_part.get("exp_name", "unknown_run").replace("/", "_")
artifact_name = CONFIG_PART.get("wandb", {}).get("artifact_name", "unknown_artifact")
artifact_type = CONFIG_PART.get("wandb", {}).get("ckpt_type", "unknown_type")

print(
    f"Using plot directory: ../Updates/plots_{run_id}/{artifact_name}:{artifact_type}"
)

In [ ]:
# ============================================================
# GENERATE AND SAVE ALL PLOTS
# ============================================================
import os
import seaborn as sns
import pandas as pd
from sklearn.metrics import roc_auc_score

# Extract run_id from artifact path for plot directory
run_id = config_part.get("exp_name", "unknown_run").replace("/", "_")
artifact_name = CONFIG_PART.get("wandb", {}).get("artifact_name", "unknown_artifact")
artifact_type = CONFIG_PART.get("wandb", {}).get("ckpt_type", "unknown_type")
plot_dir = f"../Updates/plots_{run_id}/{artifact_name}:{artifact_type}"
os.makedirs(plot_dir, exist_ok=True)
print(f"Saving plots to: {plot_dir}")


def save_fig(fig, name):
    """Save figure to plot directory."""
    filepath = os.path.join(plot_dir, f"{name}.png")
    fig.savefig(filepath, dpi=150, bbox_inches="tight")
    print(f"  Saved: {name}.png")


# ============================================================
# 1. MODEL OUTPUT DISTRIBUTION
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(
    all_outputs_after_cuts[all_labels_after_cuts == 0],
    bins=50,
    range=(0, 1),
    label="Background",
    histtype="step",
    color="red",
    density=True,
)
ax.hist(
    all_outputs_after_cuts[all_labels_after_cuts == 1],
    bins=50,
    range=(0, 1),
    label="Signal",
    histtype="step",
    color="blue",
    density=True,
)
ax.set_xlabel("Model Output Score")
ax.set_ylabel("Density")
ax.set_title("Model Output Distribution on Validation Set")
ax.legend()
save_fig(fig, "model_output_distribution")
plt.show()

# ============================================================
# 2. ROC CURVE COMPARISON
# ============================================================
from plotting_helpers import plot_roc_comparison

fig_roc = plot_roc_comparison(
    [
        ("Offline PNet", offline_roc),
        ("Offline UParT", offline_roc_upart),
        ("L1NG", l1ng_roc),
        ("L1ExtJet", l1ext_roc),
        ("Trained ParT", (fpr, tpr, roc_auc, thresholds)),
    ],
    working_point=0.1,
    return_fig=True,
)
if fig_roc:
    save_fig(fig_roc, "roc_comparison")

# ============================================================
# 3. 2D AUC HEATMAP
# ============================================================
pt_ranges = [
    (25, 100),
    (100, 200),
    (200, 300),
    (300, 400),
    (400, 500),
    (500, np.inf),
    (25, np.inf),
]
eta_ranges = [(0, 0.5), (0.5, 1.0), (1.0, 1.5), (1.5, 2.4), (0, 1.5), (0, 2.4)]


def calculate_trained_roc_2d_bins(
    pt_ranges, eta_ranges, jet_pts, jet_etas, labels, outputs
):
    auc_matrix = np.zeros((len(pt_ranges), len(eta_ranges)))
    count_matrix = np.zeros((len(pt_ranges), len(eta_ranges)))
    for j, (pt_low, pt_high) in enumerate(pt_ranges):
        for i, (eta_low, eta_high) in enumerate(eta_ranges):
            bin_mask = (
                (jet_pts >= pt_low)
                & (jet_pts < pt_high)
                & (np.abs(jet_etas) >= eta_low)
                & (np.abs(jet_etas) < eta_high)
            )
            bin_labels, bin_outputs = labels[bin_mask], outputs[bin_mask]
            count_matrix[j, i] = len(bin_labels)
            if len(np.unique(bin_labels)) < 2 or len(bin_labels) < 10:
                auc_matrix[j, i] = np.nan
            else:
                try:
                    auc_matrix[j, i] = roc_auc_score(bin_labels, bin_outputs)
                except:
                    auc_matrix[j, i] = np.nan
    return auc_matrix, count_matrix


# Use reconstructed jet kinematics
val_jet_pt_cuts = val_jet_pt[val_cuts_mask]
val_jet_eta_cuts = val_jet_eta[val_cuts_mask]

auc_matrix, count_matrix = calculate_trained_roc_2d_bins(
    pt_ranges,
    eta_ranges,
    val_jet_pt_cuts,
    val_jet_eta_cuts,
    all_labels_after_cuts,
    all_outputs_after_cuts,
)

# Plot heatmap
pt_labels = [f"[{low},{high})" for low, high in pt_ranges]
eta_labels = [f"[{low},{high})" for low, high in eta_ranges]
annot = np.empty_like(auc_matrix, dtype=object)
for i in range(auc_matrix.shape[0]):
    for j in range(auc_matrix.shape[1]):
        if np.isnan(auc_matrix[i, j]):
            annot[i, j] = f"N={int(count_matrix[i, j])}\n(N/A)"
        else:
            annot[i, j] = f"{auc_matrix[i, j]:.3f}\n(N={int(count_matrix[i, j])})"

fig, ax = plt.subplots(figsize=(12, 9))
df = pd.DataFrame(auc_matrix, index=pt_labels, columns=eta_labels)
sns.heatmap(
    df,
    annot=annot,
    fmt="",
    cmap="viridis",
    vmin=0.5,
    vmax=1.0,
    ax=ax,
    cbar_kws={"label": "AUC"},
)
ax.set_xlabel(r"$|\eta|$")
ax.set_ylabel("Jet pT [GeV]")
ax.set_title(r"Trained ParT: AUC in pT vs $|\eta|$ Bins")
plt.tight_layout()
save_fig(fig, "auc_heatmap_pt_eta")
plt.show()

# ============================================================
# AUC HEATMAP WITH BOOTSTRAP UNCERTAINTIES
# ============================================================


def calculate_auc_uncertainty_2d_bins(
    pt_ranges, eta_ranges, jet_pts, jet_etas, labels, outputs, n_boot=50
):
    """Calculate AUC with bootstrap uncertainty for each pT-eta bin."""
    auc_mat = np.zeros((len(pt_ranges), len(eta_ranges)))
    unc_mat = np.zeros((len(pt_ranges), len(eta_ranges)))
    cnt_mat = np.zeros((len(pt_ranges), len(eta_ranges)))

    for j, (pt_low, pt_high) in enumerate(pt_ranges):
        for i, (eta_low, eta_high) in enumerate(eta_ranges):
            bin_mask = (
                (jet_pts >= pt_low)
                & (jet_pts < pt_high)
                & (np.abs(jet_etas) >= eta_low)
                & (np.abs(jet_etas) < eta_high)
            )
            bin_labels = labels[bin_mask]
            bin_outputs = outputs[bin_mask]
            cnt_mat[j, i] = len(bin_labels)

            if len(np.unique(bin_labels)) < 2 or len(bin_labels) < 10:
                auc_mat[j, i] = np.nan
                unc_mat[j, i] = np.nan
            else:
                boot_aucs = []
                for _ in range(n_boot):
                    idx = np.random.choice(
                        len(bin_labels), len(bin_labels), replace=True
                    )
                    if len(np.unique(bin_labels[idx])) < 2:
                        continue
                    try:
                        boot_aucs.append(
                            roc_auc_score(bin_labels[idx], bin_outputs[idx])
                        )
                    except:
                        continue
                if len(boot_aucs) >= 10:
                    auc_mat[j, i] = np.mean(boot_aucs)
                    unc_mat[j, i] = np.std(boot_aucs)
                else:
                    try:
                        auc_mat[j, i] = roc_auc_score(bin_labels, bin_outputs)
                        unc_mat[j, i] = np.nan
                    except:
                        auc_mat[j, i] = np.nan
                        unc_mat[j, i] = np.nan
    return auc_mat, unc_mat, cnt_mat


print("Computing AUC with bootstrap uncertainties...")
auc_matrix_u, unc_matrix_u, count_matrix_u = calculate_auc_uncertainty_2d_bins(
    pt_ranges,
    eta_ranges,
    val_jet_pt_cuts,
    val_jet_eta_cuts,
    all_labels_after_cuts,
    all_outputs_after_cuts,
    n_boot=50,
)

annot_u = np.empty_like(auc_matrix_u, dtype=object)
for i in range(auc_matrix_u.shape[0]):
    for j in range(auc_matrix_u.shape[1]):
        if np.isnan(auc_matrix_u[i, j]):
            annot_u[i, j] = f"N={int(count_matrix_u[i, j])}\n(N/A)"
        elif np.isnan(unc_matrix_u[i, j]):
            annot_u[i, j] = f"{auc_matrix_u[i, j]:.3f}\n(N={int(count_matrix_u[i, j])})"
        else:
            annot_u[i, j] = (
                f"{auc_matrix_u[i, j]:.3f}±{unc_matrix_u[i, j]:.3f}\n(N={int(count_matrix_u[i, j])})"
            )

fig_unc, ax_unc = plt.subplots(figsize=(14, 10))
df_unc = pd.DataFrame(auc_matrix_u, index=pt_labels, columns=eta_labels)
sns.heatmap(
    df_unc,
    annot=annot_u,
    fmt="",
    cmap="viridis",
    vmin=0.5,
    vmax=1.0,
    ax=ax_unc,
    cbar_kws={"label": "AUC"},
    annot_kws={"fontsize": 9},
)
ax_unc.set_xlabel(r"$|\eta|$")
ax_unc.set_ylabel("Jet pT [GeV]")
ax_unc.set_title(r"Trained ParT: AUC with Bootstrap Uncertainty in pT vs $|\eta|$ Bins")
plt.tight_layout()
save_fig(fig_unc, "auc_heatmap_pt_eta_uncertainty")
plt.show()
print("AUC uncertainty heatmap saved!")

# ============================================================
# 4. CONSTITUENT FEATURE DISTRIBUTIONS
# ============================================================
signal_mask_full = (y_full == 1).numpy().flatten()
background_mask_full = (y_full == 0).numpy().flatten()
mask_np = mask_full.numpy()

for i, feat_name in enumerate(feature_names):
    fig, ax = plt.subplots(figsize=(10, 6))
    sig_vals = x_features[:, :, i].numpy()[(y_full == 1).numpy() & mask_np]
    bkg_vals = x_features[:, :, i].numpy()[(y_full == 0).numpy() & mask_np]

    ax.hist(
        sig_vals.flatten(),
        bins=50,
        range=(np.percentile(sig_vals, 1), np.percentile(sig_vals, 99)),
        histtype="step",
        label="Signal (b-jets)",
        color="blue",
        density=True,
    )
    ax.hist(
        bkg_vals.flatten(),
        bins=50,
        range=(np.percentile(bkg_vals, 1), np.percentile(bkg_vals, 99)),
        histtype="step",
        label="Background",
        color="red",
        density=True,
    )
    ax.legend()
    ax.set_xlabel(f"Constituent {feat_name}")
    ax.set_ylabel("Density")
    ax.set_title(f"Constituent {feat_name} Distribution")
    save_fig(fig, f"constituent_{feat_name.lower().replace(' ', '_')}")
    plt.close(fig)

print(f"  Saved {len(feature_names)} constituent feature plots")

# ============================================================
# 5. JET KINEMATICS COMPARISON
# ============================================================
trained_pt = jet_vectors_full.pt
trained_eta = jet_vectors_full.eta

# pT comparison
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(
    trained_pt,
    bins=100,
    range=(0, 500),
    histtype="step",
    label=f"Trained AK4 Jets (N={len(trained_pt)})",
    color="green",
    density=True,
)
ax.hist(
    l1ng_pt,
    bins=100,
    range=(0, 500),
    histtype="step",
    label=f"L1NG Jets (N={len(l1ng_pt)})",
    color="blue",
    density=True,
)
ax.hist(
    offline_pt,
    bins=100,
    range=(0, 500),
    histtype="step",
    label=f"Offline Jets (N={len(offline_pt)})",
    color="red",
    density=True,
)
ax.axvline(25, color="black", linestyle="--", alpha=0.7, label="25 GeV cut")
ax.set_xlabel("Jet $p_T$ [GeV]")
ax.set_ylabel("Density")
ax.set_title("Jet $p_T$ Distribution Comparison")
ax.legend()
save_fig(fig, "jet_pt_comparison")
plt.show()

# eta comparison
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(
    trained_eta,
    bins=100,
    range=(-3, 3),
    histtype="step",
    label="Trained AK4 Jets",
    color="green",
    density=True,
)
ax.hist(
    l1ng_eta,
    bins=100,
    range=(-3, 3),
    histtype="step",
    label="L1NG Jets",
    color="blue",
    density=True,
)
ax.hist(
    offline_eta,
    bins=100,
    range=(-3, 3),
    histtype="step",
    label="Offline Jets",
    color="red",
    density=True,
)
ax.set_xlabel(r"Jet $\eta$")
ax.set_ylabel("Density")
ax.set_title(r"Jet $\eta$ Distribution Comparison")
ax.legend()
save_fig(fig, "jet_eta_comparison")
plt.show()

# ============================================================
# 6. SIGNAL VS BACKGROUND JET KINEMATICS
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# pT
ax = axes[0]
ax.hist(
    trained_pt[signal_mask_full],
    bins=50,
    range=(0, 500),
    histtype="step",
    label=f"Signal (N={signal_mask_full.sum()})",
    color="blue",
    density=True,
)
ax.hist(
    trained_pt[background_mask_full],
    bins=50,
    range=(0, 500),
    histtype="step",
    label=f"Background (N={background_mask_full.sum()})",
    color="red",
    density=True,
)

ax.set_xlabel("Jet $p_T$ [GeV]")
ax.set_ylabel("Density")
ax.set_title("Signal vs Background Jet $p_T$")
ax.legend()

# eta
ax = axes[1]
ax.hist(
    trained_eta[signal_mask_full],
    bins=50,
    range=(-3, 3),
    histtype="step",
    label="Signal",
    color="blue",
    density=True,
)
ax.hist(
    trained_eta[background_mask_full],
    bins=50,
    range=(-3, 3),
    histtype="step",
    label="Background",
    color="red",
    density=True,
)
ax.set_xlabel(r"Jet $\eta$")
ax.set_ylabel("Density")
ax.set_title(r"Signal vs Background Jet $\eta$")
ax.legend()

plt.tight_layout()
save_fig(fig, "signal_vs_background_kinematics")
plt.show()

# ============================================================
# 7. 2D pT-eta DISTRIBUTIONS
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for ax, (data_pt, data_eta, title) in zip(
    axes,
    [
        (trained_pt, trained_eta, "Trained AK4 Jets"),
        (l1ng_pt, l1ng_eta, "L1NG Jets"),
        (offline_pt, offline_eta, "Offline Jets"),
    ],
):
    h = ax.hist2d(
        data_eta,
        data_pt,
        bins=[50, 50],
        range=[[-3, 3], [0, 300]],
        cmap="viridis",
        norm=plt.matplotlib.colors.LogNorm(),
    )
    ax.set_xlabel(r"Jet $\eta$")
    ax.set_ylabel("Jet $p_T$ [GeV]")
    ax.set_title(f"{title}: $p_T$ vs $\\eta$")
    plt.colorbar(h[3], ax=ax, label="Counts")
plt.tight_layout()
save_fig(fig, "jet_pt_eta_2d_comparison")
plt.show()

print(f"\n{'='*60}")
print(f"All plots saved to: {plot_dir}")
print(f"{'='*60}")

In [ ]:
print(f"\n--- Saving regression head analysis plot ---")
signal_mask_reg = all_labels.squeeze() == 1
jet_pt_reg = all_jet_pt_val
gen_pt_reg = all_gen_pt_val

fig_reg, axes_reg = plt.subplots(2, 3, figsize=(18, 10))

ax = axes_reg[0, 0]
ax.hist(
    jet_pt_reg[signal_mask_reg],
    bins=60,
    range=(0, 500),
    histtype="step",
    density=True,
    label="Signal",
)
ax.hist(
    jet_pt_reg[~signal_mask_reg],
    bins=60,
    range=(0, 500),
    histtype="step",
    density=True,
    label="Background",
)
ax.set_xlabel("Reco jet $p_T$ [GeV]")
ax.set_ylabel("Density")
ax.set_title("Reco jet $p_T$ distribution")
ax.legend()

ax = axes_reg[0, 1]
ax.hist(
    gen_pt_reg[signal_mask_reg],
    bins=60,
    range=(0, 500),
    histtype="step",
    density=True,
    color="C0",
)
ax.set_xlabel("Gen $p_T$ [GeV]")
ax.set_ylabel("Density")
ax.set_title("Gen $p_T$ (signal jets only)")

ax = axes_reg[0, 2]
correction_reg = gen_pt_reg[signal_mask_reg] / (jet_pt_reg[signal_mask_reg] + 1e-9)
ax.hist(
    correction_reg, bins=60, range=(0, 3), histtype="step", density=True, color="C2"
)
ax.axvline(1.0, color="gray", linestyle="--", alpha=0.7, label="No correction")
ax.set_xlabel("gen $p_T$ / reco $p_T$")
ax.set_ylabel("Density")
ax.set_title("Target correction factor (signal)")
ax.legend()

if has_regression:
    reg_pt_reg = all_reg_pt
    ax = axes_reg[1, 0]
    ax.hist(
        reg_pt_reg[signal_mask_reg],
        bins=60,
        histtype="step",
        density=True,
        label="Signal",
    )
    ax.hist(
        reg_pt_reg[~signal_mask_reg],
        bins=60,
        histtype="step",
        density=True,
        label="Background",
    )
    ax.set_xlabel("Regression output")
    ax.set_ylabel("Density")
    ax.set_title("Regression head output")
    ax.legend()

    ax = axes_reg[1, 1]
    true_corr_reg = gen_pt_reg[signal_mask_reg] / (jet_pt_reg[signal_mask_reg] + 1e-9)
    pred_corr_reg = reg_pt_reg[signal_mask_reg]
    ax.scatter(true_corr_reg, pred_corr_reg, s=1, alpha=0.3)
    lim = max(true_corr_reg.max(), pred_corr_reg.max()) * 1.05
    ax.plot([0, lim], [0, lim], "r--", alpha=0.5, label="Ideal")
    ax.set_xlabel("True correction")
    ax.set_ylabel("Predicted correction")
    ax.set_title("Predicted vs true correction")
    ax.legend()

    ax = axes_reg[1, 2]
    corrected_pt_reg = jet_pt_reg[signal_mask_reg] * pred_corr_reg
    ax.scatter(
        gen_pt_reg[signal_mask_reg], corrected_pt_reg, s=1, alpha=0.3, label="Corrected"
    )
    ax.scatter(
        gen_pt_reg[signal_mask_reg],
        jet_pt_reg[signal_mask_reg],
        s=1,
        alpha=0.1,
        color="gray",
        label="Uncorrected",
    )
    ax.plot([0, 500], [0, 500], "r--", alpha=0.5)
    ax.set_xlabel("Gen $p_T$ [GeV]")
    ax.set_ylabel("Jet $p_T$ [GeV]")
    ax.set_xlim(0, 500)
    ax.set_ylim(0, 500)
    ax.set_title("Corrected vs uncorrected $p_T$")
    ax.legend()
else:
    for i in range(3):
        axes_reg[1, i].text(
            0.5,
            0.5,
            "No regression head",
            ha="center",
            va="center",
            transform=axes_reg[1, i].transAxes,
        )
        axes_reg[1, i].set_axis_off()

plt.tight_layout()
save_fig(fig_reg, "regression_head_analysis")
plt.show()

print(f"\nAll regression plots saved to: {plot_dir}")

In [ ]:
lim = 500
fig, ax = plt.subplots()
ax.scatter(
    gen_pt_reg[signal_mask_reg],
    jet_pt_reg[signal_mask_reg],
    s=1,
    alpha=0.1,
    color="black",
    label="Uncorrected",
)
ax.plot([0, lim], [0, lim], "r--", alpha=0.5)
ax.set_xlabel("Gen $p_T$ [GeV]")
ax.set_ylabel("Jet $p_T$ [GeV]")
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_title("Uncorrected $p_T$")
ax.legend()
plt.show()

fig, ax = plt.subplots()
corrected_pt_reg = jet_pt_reg[signal_mask_reg] * pred_corr_reg
ax.scatter(
    gen_pt_reg[signal_mask_reg], corrected_pt_reg, s=1, alpha=0.3, label="Corrected"
)
ax.plot([0, lim], [0, lim], "r--", alpha=0.5)
ax.set_xlabel("Gen $p_T$ [GeV]")
ax.set_ylabel("Jet $p_T$ [GeV]")
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_title("Corrected $p_T$")
ax.legend()
plt.show()

## Jet $p_T$ Resolution Analysis

Compute and compare the jet $p_T$ resolution **before** and **after** the regression correction.

**Predicted resolution** from the quantile regression head:

$$\sigma_{p_T} = \tfrac{1}{2}\,(q_{84} - q_{16})\;\times\;\text{jet\_pt}\;\times\;\text{jet\_scale}$$

where $q_{16}$, $q_{84}$ are the predicted 16th/84th percentiles of the $p_T$ response, and jet\_scale is the regression head output.

**Calculated resolution** is the Gaussian $\sigma$ of the $p_T$ response ($\text{reco}\;p_T / \text{gen}\;p_T$) fitted in bins of gen $p_T$ and $\eta$ — computed separately before (uncorrected) and after (corrected) the $p_T$ correction.

In [ ]:
# ============================================================
# JET pT RESOLUTION: PREDICTED (QUANTILE HEAD) vs CALCULATED
# ============================================================
from scipy.optimize import curve_fit


def gaussian(x, mu, sigma, A):
    """Simple Gaussian for fitting response distributions."""
    return A * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def fit_response_in_bin(response_values, bins=np.linspace(0, 2, 51)):
    """Fit a Gaussian to the response distribution in a single bin."""
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    counts, _ = np.histogram(response_values, bins=bins)
    initial_guess = [1.0, 0.1, np.max(counts)]
    try:
        popt, pcov = curve_fit(
            gaussian, bin_centers, counts, absolute_sigma=True, p0=initial_guess
        )
        return popt, pcov  # (mu, sigma, A), covariance
    except RuntimeError:
        return (np.nan, np.nan, np.nan), np.zeros((3, 3))


def get_resolution_vs_var(gen_var, pt_response, var_bins):
    """
    Compute Gaussian mu (scale) and sigma (resolution) of the pT response
    in bins of a given variable (gen_pt or gen_eta).
    Returns bin_centers, mus, sigmas, mu_errs, sigma_errs.
    """
    bin_centers = 0.5 * (var_bins[1:] + var_bins[:-1])
    mus, sigmas = [], []
    mu_errs, sigma_errs = [], []
    for i in range(len(var_bins) - 1):
        mask = (gen_var > var_bins[i]) & (gen_var <= var_bins[i + 1])
        vals = pt_response[mask]
        if len(vals) > 20:
            (mu, sigma, A), cov = fit_response_in_bin(vals)
            mus.append(mu if not np.isnan(mu) else np.nan)
            sigmas.append(abs(sigma) if not np.isnan(sigma) else np.nan)
            mu_errs.append(np.sqrt(cov[0, 0]) if cov[0, 0] > 0 else 0.0)
            sigma_errs.append(np.sqrt(cov[1, 1]) if cov[1, 1] > 0 else 0.0)
        else:
            mus.append(np.nan)
            sigmas.append(np.nan)
            mu_errs.append(0.0)
            sigma_errs.append(0.0)
    return (
        bin_centers,
        np.array(mus),
        np.array(sigmas),
        np.array(mu_errs),
        np.array(sigma_errs),
    )


# --- Select signal jets only (background has gen_pt = 0) ---
signal_mask_res = all_labels.squeeze() == 1
jet_pt_sig = all_jet_pt_val[signal_mask_res]
gen_pt_sig = all_gen_pt_val[signal_mask_res]
gen_eta_sig = val_jet_eta[signal_mask_res]  # use reco eta as proxy (matched)

# --- Uncorrected (raw) pT response ---
raw_response = jet_pt_sig / (gen_pt_sig + 1e-9)

# --- Corrected pT response (using regression head) ---
if has_regression:
    pred_scale = all_reg_pt[signal_mask_res]
    corrected_pt = jet_pt_sig * pred_scale
    corr_response = corrected_pt / (gen_pt_sig + 1e-9)
else:
    corrected_pt = jet_pt_sig
    corr_response = raw_response
    pred_scale = np.ones_like(jet_pt_sig)

# --- Predicted resolution from quantile head ---
if has_quantile:
    q16 = all_quantiles[signal_mask_res, 0]
    q84 = all_quantiles[signal_mask_res, 1]
    # Resolution = (1/2) * (q84 - q16) * jet_pt * jet_scale
    predicted_resolution_abs = 0.5 * (q84 - q16) * jet_pt_sig * pred_scale
    # Relative resolution (dimensionless) = (1/2) * (q84 - q16)
    predicted_resolution_rel = 0.5 * (q84 - q16)
    print(
        f"Predicted resolution (absolute) range: "
        f"{predicted_resolution_abs.min():.2f} – {predicted_resolution_abs.max():.2f} GeV"
    )
    print(
        f"Predicted resolution (relative)  range: "
        f"{predicted_resolution_rel.min():.4f} – {predicted_resolution_rel.max():.4f}"
    )

# --- Binning ---
pt_bins = np.linspace(25, 500, 30)
eta_bins = np.linspace(-2.4, 2.4, 20)

# Calculated resolution vs gen pT (before and after correction)
bc_pt, mu_raw_pt, sig_raw_pt, mu_raw_pt_err, sig_raw_pt_err = get_resolution_vs_var(
    gen_pt_sig, raw_response, pt_bins
)
bc_pt, mu_corr_pt, sig_corr_pt, mu_corr_pt_err, sig_corr_pt_err = get_resolution_vs_var(
    gen_pt_sig, corr_response, pt_bins
)

# Calculated resolution vs eta (before and after correction)
bc_eta, mu_raw_eta, sig_raw_eta, mu_raw_eta_err, sig_raw_eta_err = (
    get_resolution_vs_var(gen_eta_sig, raw_response, eta_bins)
)
bc_eta, mu_corr_eta, sig_corr_eta, mu_corr_eta_err, sig_corr_eta_err = (
    get_resolution_vs_var(gen_eta_sig, corr_response, eta_bins)
)

# Predicted (quantile) resolution averaged in bins
if has_quantile:

    def avg_in_bins(gen_var, values, var_bins):
        """Average values in bins of gen_var with std-of-mean error."""
        bc = 0.5 * (var_bins[1:] + var_bins[:-1])
        means, errs = [], []
        for i in range(len(var_bins) - 1):
            m = (gen_var > var_bins[i]) & (gen_var <= var_bins[i + 1])
            v = values[m]
            if len(v) > 5:
                means.append(np.mean(v))
                errs.append(np.std(v) / np.sqrt(len(v)))
            else:
                means.append(np.nan)
                errs.append(0.0)
        return bc, np.array(means), np.array(errs)

    _, pred_res_vs_pt, pred_res_vs_pt_err = avg_in_bins(
        gen_pt_sig, predicted_resolution_rel, pt_bins
    )
    _, pred_res_vs_eta, pred_res_vs_eta_err = avg_in_bins(
        gen_eta_sig, predicted_resolution_rel, eta_bins
    )

print("Resolution computation complete.")

In [ ]:
# ============================================================
# PLOT 1: pT Response Distributions (Before vs After Correction)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1a. Overall response distribution
ax = axes[0]
ax.hist(
    raw_response,
    bins=np.linspace(0, 2, 81),
    histtype="step",
    density=True,
    label="Uncorrected",
    color="C0",
    linewidth=1.5,
)
if has_regression:
    ax.hist(
        corr_response,
        bins=np.linspace(0, 2, 81),
        histtype="step",
        density=True,
        label="Corrected (regression)",
        color="C1",
        linewidth=1.5,
    )
ax.axvline(1.0, color="gray", linestyle="--", alpha=0.7, label="Ideal (1.0)")
ax.set_xlabel("$p_T$ Response (reco $p_T$ / gen $p_T$)")
ax.set_ylabel("Density")
ax.set_title("Jet $p_T$ Response Distribution (Signal Jets)")
ax.legend()
ax.set_xlim(0, 2)

# 1b. Response in selected gen pT bins
ax = axes[1]
pt_slices = [(50, 100), (100, 200), (200, 300), (300, 500)]
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(pt_slices)))
for (pt_lo, pt_hi), color in zip(pt_slices, colors):
    mask = (gen_pt_sig >= pt_lo) & (gen_pt_sig < pt_hi)
    if has_regression:
        ax.hist(
            corr_response[mask],
            bins=np.linspace(0, 2, 61),
            histtype="step",
            density=True,
            color=color,
            linewidth=1.5,
            label=f"Corrected [{pt_lo},{pt_hi}) GeV",
        )
    # ax.hist(
    #     raw_response[mask],
    #     bins=np.linspace(0, 2, 61),
    #     histtype="step",
    #     density=True,
    #     color=color,
    #     linewidth=1.0,
    #     linestyle="--",
    #     label=f"Uncorrected [{pt_lo},{pt_hi}) GeV",
    # )
ax.axvline(1.0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("$p_T$ Response")
ax.set_ylabel("Density")
ax.set_title("$p_T$ Response in gen $p_T$ Bins")
ax.legend(fontsize="x-small", ncol=1)
ax.set_xlim(0, 2)

plt.tight_layout()
save_fig(fig, "pt_response_distributions")
plt.show()

In [ ]:
fig, ax = plt.subplots()
pt_slices = [(50, 100), (100, 200), (200, 300), (300, 500)]
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(pt_slices)))
for (pt_lo, pt_hi), color in zip(pt_slices, colors):
    mask = (gen_pt_sig >= pt_lo) & (gen_pt_sig < pt_hi)
    if has_regression:
        ax.hist(
            raw_response[mask],
            bins=np.linspace(0, 2, 61),
            histtype="step",
            density=True,
            color=color,
            linewidth=1.0,
            linestyle="--",
            label=f"Uncorrected [{pt_lo},{pt_hi}) GeV",
        )
ax.axvline(1.0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("$p_T$ Response")
ax.set_ylabel("Density")
ax.set_title("$p_T$ Response in gen $p_T$ Bins")
ax.legend(fontsize="x-small", ncol=1)
ax.set_xlim(0, 2)

plt.tight_layout()
# save_fig(fig, "pt_response_distributions")
plt.show()

In [ ]:
# ============================================================
# PLOT 2: Scale and Resolution vs gen pT (Before & After Correction)
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# --- Top panel: Scale (mean of response) vs gen pT ---
ax = axes[0]
ax.errorbar(
    bc_pt,
    mu_raw_pt,
    yerr=mu_raw_pt_err,
    marker="o",
    linestyle="-",
    capsize=4,
    label="Uncorrected",
    color="C0",
)
if has_regression:
    ax.errorbar(
        bc_pt,
        mu_corr_pt,
        yerr=mu_corr_pt_err,
        marker="s",
        linestyle="-",
        capsize=4,
        label="Corrected (regression)",
        color="C1",
    )
ax.axhline(1.0, color="black", linestyle="-", linewidth=0.8)
ax.set_ylabel("Jet $p_T$ Scale (Mean of Response)")
ax.set_title("Jet $p_T$ Scale & Resolution vs. Generated $p_T$ (Signal Jets)")
ax.legend()
ax.set_ylim(0.5, 1.5)

# --- Bottom panel: Resolution (sigma) vs gen pT ---
ax = axes[1]
ax.errorbar(
    bc_pt,
    sig_raw_pt,
    yerr=sig_raw_pt_err,
    marker="o",
    linestyle="-",
    capsize=4,
    label="Calculated (uncorrected)",
    color="C0",
)
if has_regression:
    ax.errorbar(
        bc_pt,
        sig_corr_pt,
        yerr=sig_corr_pt_err,
        marker="s",
        linestyle="-",
        capsize=4,
        label="Calculated (corrected)",
        color="C1",
    )
if has_quantile:
    ax.errorbar(
        bc_pt,
        pred_res_vs_pt,
        yerr=pred_res_vs_pt_err,
        marker="^",
        linestyle="--",
        capsize=4,
        label="Predicted (quantile head)",
        color="C2",
    )
ax.set_xlabel("Generated Jet $p_T$ [GeV]")
ax.set_ylabel("Resolution ($\\sigma$ of Response)")
ax.legend()
ax.set_ylim(0)

plt.tight_layout()
save_fig(fig, "resolution_vs_gen_pt")
plt.show()

In [ ]:
# ============================================================
# PLOT 3: Scale and Resolution vs eta (Before & After Correction)
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# --- Top: Scale vs eta ---
ax = axes[0]
ax.errorbar(
    bc_eta,
    mu_raw_eta,
    yerr=mu_raw_eta_err,
    marker="o",
    linestyle="-",
    capsize=4,
    label="Uncorrected",
    color="C0",
)
if has_regression:
    ax.errorbar(
        bc_eta,
        mu_corr_eta,
        yerr=mu_corr_eta_err,
        marker="s",
        linestyle="-",
        capsize=4,
        label="Corrected (regression)",
        color="C1",
    )
ax.axhline(1.0, color="black", linestyle="-", linewidth=0.8)
ax.set_ylabel("Jet $p_T$ Scale (Mean of Response)")
ax.set_title("Jet $p_T$ Scale & Resolution vs. Jet $\\eta$ (Signal Jets)")
ax.legend()
ax.set_ylim(0.5, 1.5)

# --- Bottom: Resolution vs eta ---
ax = axes[1]
ax.errorbar(
    bc_eta,
    sig_raw_eta,
    yerr=sig_raw_eta_err,
    marker="o",
    linestyle="-",
    capsize=4,
    label="Calculated (uncorrected)",
    color="C0",
)
if has_regression:
    ax.errorbar(
        bc_eta,
        sig_corr_eta,
        yerr=sig_corr_eta_err,
        marker="s",
        linestyle="-",
        capsize=4,
        label="Calculated (corrected)",
        color="C1",
    )
if has_quantile:
    ax.errorbar(
        bc_eta,
        pred_res_vs_eta,
        yerr=pred_res_vs_eta_err,
        marker="^",
        linestyle="--",
        capsize=4,
        label="Predicted (quantile head)",
        color="C2",
    )
ax.set_xlabel("Jet $\\eta$")
ax.set_ylabel("Resolution ($\\sigma$ of Response)")
ax.legend()
ax.set_ylim(0)

plt.tight_layout()
save_fig(fig, "resolution_vs_eta")
plt.show()

In [ ]:
# ============================================================
# PLOT 4: 2D Response Maps (Before & After) + Predicted Resolution
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

# 4a. Uncorrected pT response vs gen pT
ax = axes[0]
h0 = ax.hist2d(
    gen_pt_sig,
    raw_response,
    bins=[np.linspace(25, 500, 51), np.linspace(0, 2, 51)],
    cmap="viridis",
    cmin=1,
)
ax.axhline(1.0, color="red", linestyle="--", alpha=0.7)
ax.set_xlabel("Generated $p_T$ [GeV]")
ax.set_ylabel("$p_T$ Response (reco / gen)")
ax.set_title("Uncorrected $p_T$ Response")
plt.colorbar(h0[3], ax=ax, label="Counts")

# 4b. Corrected pT response vs gen pT
ax = axes[1]
h1 = ax.hist2d(
    gen_pt_sig,
    corr_response,
    bins=[np.linspace(25, 500, 51), np.linspace(0, 2, 51)],
    cmap="viridis",
    cmin=1,
)
ax.axhline(1.0, color="red", linestyle="--", alpha=0.7)
ax.set_xlabel("Generated $p_T$ [GeV]")
ax.set_ylabel("$p_T$ Response (corrected / gen)")
ax.set_title("Corrected $p_T$ Response")
plt.colorbar(h1[3], ax=ax, label="Counts")

# 4c. Predicted resolution vs gen pT (scatter)
ax = axes[2]
if has_quantile:
    sc = ax.scatter(
        gen_pt_sig,
        predicted_resolution_rel,
        c=jet_pt_sig,
        s=1,
        alpha=0.3,
        cmap="plasma",
    )
    plt.colorbar(sc, ax=ax, label="Reco Jet $p_T$ [GeV]")
    ax.set_xlabel("Generated $p_T$ [GeV]")
    ax.set_ylabel("Predicted Relative Resolution")
    ax.set_title("Quantile-Predicted Resolution vs. Gen $p_T$")
    ax.set_ylim(0, 0.5)
else:
    ax.text(
        0.5,
        0.5,
        "No quantile head",
        ha="center",
        va="center",
        transform=ax.transAxes,
        fontsize=14,
    )
    ax.set_axis_off()

plt.tight_layout()
save_fig(fig, "response_2d_maps")
plt.show()

In [ ]:
# ============================================================
# PLOT 5: Gaussian fits in selected gen pT slices
# ============================================================
pt_slices_fit = [(25, 100), (100, 150), (200, 250), (300, 400)]
n_slices = len(pt_slices_fit)
fig, axes = plt.subplots(2, n_slices, figsize=(5 * n_slices, 10))

response_bins = np.linspace(0, 2, 61)
x_fit = np.linspace(0, 2, 200)

for j, (pt_lo, pt_hi) in enumerate(pt_slices_fit):
    mask = (gen_pt_sig >= pt_lo) & (gen_pt_sig < pt_hi)

    # --- Top row: Uncorrected ---
    ax = axes[0, j]
    vals_raw = raw_response[mask]
    counts_raw, _ = np.histogram(vals_raw, bins=response_bins)
    (mu_r, sig_r, A_r), _ = fit_response_in_bin(vals_raw, response_bins)
    ax.stairs(counts_raw, response_bins, color="C0", linewidth=1.5, label="Uncorrected")
    if not np.isnan(mu_r):
        ax.plot(
            x_fit,
            gaussian(x_fit, mu_r, sig_r, A_r),
            "C0--",
            label=f"$\\mu$={mu_r:.3f}, $\\sigma$={abs(sig_r):.3f}",
        )
    ax.set_title(f"gen $p_T$ [{pt_lo},{pt_hi}) GeV\n(Uncorrected)")
    ax.set_xlabel("$p_T$ Response")
    ax.set_ylabel("Counts")
    ax.legend(fontsize="x-small")

    # --- Bottom row: Corrected ---
    ax = axes[1, j]
    if has_regression:
        vals_corr = corr_response[mask]
        counts_corr, _ = np.histogram(vals_corr, bins=response_bins)
        (mu_c, sig_c, A_c), _ = fit_response_in_bin(vals_corr, response_bins)
        ax.stairs(
            counts_corr, response_bins, color="C1", linewidth=1.5, label="Corrected"
        )
        if not np.isnan(mu_c):
            ax.plot(
                x_fit,
                gaussian(x_fit, mu_c, sig_c, A_c),
                "C1--",
                label=f"$\\mu$={mu_c:.3f}, $\\sigma$={abs(sig_c):.3f}",
            )
        ax.set_title(f"gen $p_T$ [{pt_lo},{pt_hi}) GeV\n(Corrected)")
    else:
        ax.text(
            0.5,
            0.5,
            "No regression head",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
    ax.set_xlabel("$p_T$ Response")
    ax.set_ylabel("Counts")
    ax.legend(fontsize="x-small")

plt.tight_layout()
save_fig(fig, "response_gaussian_fits_pt_slices")
plt.show()

In [ ]:
# ============================================================
# PLOT 6: Absolute pT Resolution (GeV) and Improvement Summary
# ============================================================
if has_quantile:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # 6a. Absolute resolution: predicted vs calculated, before & after
    ax = axes[0]
    # Calculated absolute resolution = sigma_relative * gen_pt for each bin
    abs_raw_res = sig_raw_pt * bc_pt
    abs_corr_res = sig_corr_pt * bc_pt
    ax.errorbar(
        bc_pt,
        abs_raw_res,
        marker="o",
        linestyle="-",
        capsize=4,
        label="Calculated (uncorrected)",
        color="C0",
    )
    if has_regression:
        ax.errorbar(
            bc_pt,
            abs_corr_res,
            marker="s",
            linestyle="-",
            capsize=4,
            label="Calculated (corrected)",
            color="C1",
        )
    _, pred_abs_vs_pt, pred_abs_vs_pt_err = avg_in_bins(
        gen_pt_sig, predicted_resolution_abs, pt_bins
    )
    ax.errorbar(
        bc_pt,
        pred_abs_vs_pt,
        yerr=pred_abs_vs_pt_err,
        marker="^",
        linestyle="--",
        capsize=4,
        label="Predicted (quantile head)",
        color="C2",
    )
    ax.set_xlabel("Generated Jet $p_T$ [GeV]")
    ax.set_ylabel("Absolute $p_T$ Resolution [GeV]")
    ax.set_title("Absolute Jet $p_T$ Resolution vs. Gen $p_T$")
    ax.legend()
    ax.set_ylim(0)

    # 6b. Fractional improvement: (sigma_raw - sigma_corr) / sigma_raw
    ax = axes[1]
    if has_regression:
        valid = (~np.isnan(sig_raw_pt)) & (~np.isnan(sig_corr_pt)) & (sig_raw_pt > 0)
        improvement = (sig_raw_pt[valid] - sig_corr_pt[valid]) / sig_raw_pt[valid] * 100
        ax.bar(
            bc_pt[valid],
            improvement,
            width=np.diff(pt_bins).mean() * 0.7,
            color="C3",
            alpha=0.7,
        )
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_xlabel("Generated Jet $p_T$ [GeV]")
        ax.set_ylabel("Resolution Improvement [%]")
        ax.set_title("Resolution Improvement from Regression Correction")
        # Print summary
        mean_improvement = np.nanmean(improvement)
        ax.text(
            0.05,
            0.95,
            f"Mean improvement: {mean_improvement:.1f}%",
            transform=ax.transAxes,
            va="top",
            fontsize=12,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
        )
    else:
        ax.text(
            0.5,
            0.5,
            "No regression head",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )

    plt.tight_layout()
    save_fig(fig, "absolute_resolution_and_improvement")
    plt.show()
else:
    print("Skipping absolute resolution plots (no quantile regression head).")

print(f"\nAll resolution plots saved to: {plot_dir}")

## Di-Higgs Mass Reconstruction with ParT B-Tag Scores

Load L1 PF and PUPPI candidates from ROOT files, cluster with FastJet, extract constituent features, score each jet with the trained ParT model, apply a working-point b-tag cut, correct jet $p_T$ with the regression head, and reconstruct $m_{H_1}$, $m_{H_2}$, $m_{HH}$ via $D_{HH}$ minimisation.

In [ ]:
# ============================================================
# DI-HIGGS MASS RECONSTRUCTION WITH TRAINED ParT
# ============================================================
# Step 1: Load ROOT data, cluster L1 PF/PUPPI candidates with FastJet
# Step 2: Extract ParT features per jet, run inference
# Step 3: Apply working-point b-tag cut + pT regression correction
# Step 4: D_HH pairing → mH1, mH2, mHH
# ============================================================

import fastjet
from itertools import combinations
from make_dataset import cluster_candidates
from data_loading_helpers import (
    load_and_prepare_data,
    select_gen_b_quarks_from_higgs,
    apply_custom_cuts,
    one_hot_encode_l1_puppi,
)
from analysis_helpers import get_purity_mask_cross_matched

# --- Configuration ---
apply_pt_correction = True  # Whether to apply the pT regression correction
WP_SELECTION = "tight"  # Choose from: "tight", "medium", "loose"
wp_index = {"tight": 0, "medium": 1, "loose": 2}[WP_SELECTION]
PART_BTAG_THRESHOLD = part_wps[wp_index]
print(
    f"Working point: {WP_SELECTION} → ParT b-tag threshold = {PART_BTAG_THRESHOLD:.4f}"
)

# Use the same dataset config the model was trained on
dataset_used = config_part.get("training", {}).get("data", {}).get("use_dataset", "pf")
if dataset_used == "pf":
    collection_key = "l1extpf"
elif dataset_used == "puppi":
    collection_key = "l1extpuppi"
else:
    collection_key = "l1barrelextpf"
print(f"Model was trained on: {dataset_used} → clustering {collection_key}")

# Use the hh4b ROOT files (same source as read-data.ipynb)
# The config file_pattern points to hh4b/ which has all collections including L1BarrelExtPf/Puppi
root_data_pattern = config["file_pattern"]  # from hh-bbbb-obj-config.json
collection_name = config[collection_key]["collection_name"]
print(f"ROOT data: {root_data_pattern}")
print(f"Collection: {collection_name}")

# --- Step 1: Load ROOT data and cluster ---
print("\n--- Loading ROOT data and clustering L1 candidates ---")
dihiggs_events = load_and_prepare_data(
    root_data_pattern,
    config["tree_name"],
    [collection_name, "GenPart"],
    max_events=config["max_events"],
    correct_pt=False,
    CONFIG=config,
)

# Cluster candidates into jets using anti-kT (R=0.4)
print(f"\nClustering {collection_key} candidates...")
clustered_jets = cluster_candidates(
    dihiggs_events, config, collection_key, dist_param=0.4
)

# Sort jets by pT (descending)
sorted_indices = ak.argsort(clustered_jets.pt, axis=1, ascending=False)
l1_clustered = clustered_jets[sorted_indices]

# Get constituents (already attached by cluster_candidates)
matched_cands = l1_clustered.constituents
const_pt_sort = ak.argsort(matched_cands.pt, axis=2, ascending=False)
matched_cands = matched_cands[const_pt_sort]

n_jets_total = ak.sum(ak.num(l1_clustered, axis=1))
n_events_total = len(l1_clustered)
print(f"Clustered {n_jets_total} jets across {n_events_total} events")

# --- Step 2: Extract features and run ParT inference ---
print("\n--- Extracting features and running ParT inference ---")
n_constituents = config_part.get("input_dim_n_constituents", 16)
# Infer from model input_dim expected shape
# Features: mass, pt, eta, phi, dxy, z0, charge, log_pt_rel, deta, dphi, puppiWeight, log_dr, 5×one_hot = 17
n_constituents = all_constituents.shape[1]  # Use same as training

j_pt = l1_clustered.pt[:, :, None]
j_eta = l1_clustered.eta[:, :, None]
j_phi = l1_clustered.phi[:, :, None]

m_pt = matched_cands.vector.pt
m_eta = matched_cands.vector.eta
m_phi = matched_cands.vector.phi
m_mass = matched_cands.vector.mass
m_dxy = matched_cands.dxy
m_z0 = matched_cands.z0
m_charge = matched_cands.charge
m_w = matched_cands.puppiWeight
m_id = matched_cands.id

log_pt_rel = np.log(np.maximum(m_pt, 1e-3) / np.maximum(j_pt, 1e-3))
deta = m_eta - j_eta
dphi = m_phi - j_phi
dphi = (dphi + np.pi) % (2 * np.pi) - np.pi
log_dr = np.log(np.maximum(np.sqrt(deta**2 + dphi**2), 1e-3))


def pad_and_fill(arr, target=n_constituents):
    return ak.fill_none(ak.pad_none(arr, target, axis=2, clip=True), 0.0)


feature_list = [
    pad_and_fill(m_mass),
    pad_and_fill(m_pt),
    pad_and_fill(m_eta),
    pad_and_fill(m_phi),
    pad_and_fill(m_dxy),
    pad_and_fill(m_z0),
    pad_and_fill(m_charge),
    pad_and_fill(log_pt_rel),
    pad_and_fill(deta),
    pad_and_fill(dphi),
    pad_and_fill(m_w),
    pad_and_fill(log_dr),
    pad_and_fill(m_id),
]

# Track jet counts per event BEFORE flattening
n_jets_per_event = ak.num(l1_clustered, axis=1)
# Also track constituent counts for particle mask
n_actual_constituents = ak.num(matched_cands, axis=2)
n_actual_flat = ak.to_numpy(ak.flatten(n_actual_constituents, axis=1))

# Flatten to (n_jets, n_constituents, n_features)
x_ini = np.stack([ak.to_numpy(ak.flatten(f, axis=1)) for f in feature_list], axis=-1)
flat_ids = x_ini[..., -1]
one_hot_ids = one_hot_encode_l1_puppi(flat_ids, n_classes=5)
X_all = np.concatenate([x_ini[..., :-1], one_hot_ids], axis=-1)

# Particle mask
particle_mask_all = np.zeros((X_all.shape[0], n_constituents), dtype=bool)
for i in range(X_all.shape[0]):
    n_real = min(n_actual_flat[i], n_constituents)
    particle_mask_all[i, :n_real] = True

# Also compute jet_pt from constituent sum (same as training)
const_vectors = vector.array(
    {
        "pt": x_ini[:, :, 1],
        "eta": x_ini[:, :, 2],
        "phi": x_ini[:, :, 3],
        "mass": x_ini[:, :, 0],
    }
)
jet_4vecs_flat = const_vectors.sum(axis=1)
flat_jet_pt = jet_4vecs_flat.pt
flat_jet_eta = jet_4vecs_flat.eta
flat_jet_phi = jet_4vecs_flat.phi
flat_jet_mass = jet_4vecs_flat.mass

print(f"Feature matrix shape: {X_all.shape}")
print(f"Running ParT inference on {X_all.shape[0]} jets...")

# Run model inference in batches
batch_size = config_part.get("training", {}).get("batch_size", 512)
all_scores = []
all_reg_corrections = []

model.eval()
with torch.no_grad():
    for start in tqdm(range(0, len(X_all), batch_size)):
        end = min(start + batch_size, len(X_all))
        X_batch = torch.tensor(X_all[start:end], dtype=torch.float32).to(device)
        mask_batch = torch.tensor(particle_mask_all[start:end], dtype=torch.bool).to(
            device
        )

        out = model(X_batch, particle_mask=mask_batch)

        cls_logits = out["classification"]
        scores = torch.nn.functional.sigmoid(cls_logits).squeeze().cpu().numpy()
        all_scores.append(scores)

        if "pt" in out:
            reg = out["pt"].squeeze().cpu().numpy()
            all_reg_corrections.append(reg)

all_scores = np.concatenate(all_scores)
has_reg = len(all_reg_corrections) > 0
if has_reg:
    all_reg_corrections = np.concatenate(all_reg_corrections)
    print(
        f"Regression corrections range: {all_reg_corrections.min():.3f} – {all_reg_corrections.max():.3f}"
    )

print(f"Score range: {all_scores.min():.4f} – {all_scores.max():.4f}")
print(
    f"Jets above {WP_SELECTION} WP ({PART_BTAG_THRESHOLD:.4f}): {np.sum(all_scores > PART_BTAG_THRESHOLD)}/{len(all_scores)}"
)

# --- Step 3: Apply b-tag cut and pT correction, then rebuild event structure ---
print("\n--- Applying b-tag cut and pT regression correction ---")

# Apply pT correction: corrected_pt = jet_pt * reg_pt
if has_reg and apply_pt_correction:
    corrected_pt = flat_jet_pt * all_reg_corrections
    print(
        f"pT correction applied: mean correction factor = {all_reg_corrections.mean():.3f}"
    )
else:
    corrected_pt = flat_jet_pt
    print("No regression head — using uncorrected pT")

# Build corrected jet 4-vectors (flat)
corrected_jet_vecs_flat = vector.array(
    {
        "pt": corrected_pt,
        "eta": flat_jet_eta,
        "phi": flat_jet_phi,
        "mass": flat_jet_mass
        * (corrected_pt / (flat_jet_pt + 1e-9)),  # scale mass by same factor
    }
)

# Unflatten back to event structure: (events, jets)
n_jets_np = ak.to_numpy(n_jets_per_event)
cumulative = np.concatenate([[0], np.cumsum(n_jets_np)])

# Build per-event lists
event_jet_pts = []
event_jet_etas = []
event_jet_phis = []
event_jet_masses = []
event_scores = []

for i in range(len(n_jets_np)):
    start_idx = cumulative[i]
    end_idx = cumulative[i + 1]
    event_jet_pts.append(corrected_jet_vecs_flat.pt[start_idx:end_idx])
    event_jet_etas.append(corrected_jet_vecs_flat.eta[start_idx:end_idx])
    event_jet_phis.append(corrected_jet_vecs_flat.phi[start_idx:end_idx])
    event_jet_masses.append(corrected_jet_vecs_flat.mass[start_idx:end_idx])
    event_scores.append(all_scores[start_idx:end_idx])

# Convert to awkward arrays with event structure
scored_jets = ak.zip(
    {
        "pt": ak.Array(event_jet_pts),
        "eta": ak.Array(event_jet_etas),
        "phi": ak.Array(event_jet_phis),
        "mass": ak.Array(event_jet_masses),
        "btag_score": ak.Array(event_scores),
    }
)
scored_jets["vector"] = ak.zip(
    {
        "pt": scored_jets.pt,
        "eta": scored_jets.eta,
        "phi": scored_jets.phi,
        "mass": scored_jets.mass,
    },
    with_name="Momentum4D",
)

print(
    f"Scored jets per event: mean={ak.mean(ak.num(scored_jets)):.1f}, max={ak.max(ak.num(scored_jets))}"
)

# --- Step 4: Di-Higgs reconstruction ---
print("\n--- Running di-Higgs reconstruction ---")

# Gen b-quarks for purity matching
dihiggs_gen_b = select_gen_b_quarks_from_higgs(dihiggs_events)
dihiggs_gen_b = dihiggs_gen_b[
    (dihiggs_gen_b.pt > config["gen"]["pt_cut"])
    & (abs(dihiggs_gen_b.eta) < config["gen"]["eta_cut"])
]

# Sort jets by b-tag score (descending) and take top 4
jets_btag_sorted = scored_jets[ak.argsort(scored_jets.btag_score, ascending=False)]

# Apply b-tag threshold: only keep jets above WP
jets_tagged = jets_btag_sorted[jets_btag_sorted.btag_score > PART_BTAG_THRESHOLD]
print(f"Jets per event after b-tag cut: mean={ak.mean(ak.num(jets_tagged)):.1f}")

# For reconstruction, need >=4 tagged jets
has_4_tagged = ak.num(jets_tagged) >= 4
sig_jets_all = jets_tagged[has_4_tagged][:, :4]  # Top 4 tagged jets

print(f"Events with >=4 tagged jets: {ak.sum(has_4_tagged)}/{len(jets_tagged)}")

# Cross-match for signal purity
gen_b_for_match = dihiggs_gen_b[has_4_tagged]
dr_reco = sig_jets_all[:, :, None].vector.deltaR(gen_b_for_match[:, None, :].vector)
idx_gen_for_reco = ak.argmin(dr_reco, axis=2)
min_dr_reco = ak.fill_none(ak.min(dr_reco, axis=2), np.inf)

dr_gen = gen_b_for_match[:, :, None].vector.deltaR(sig_jets_all[:, None, :].vector)
idx_reco_for_gen = ak.argmin(dr_gen, axis=2)

back_check = idx_reco_for_gen[idx_gen_for_reco]
reco_idx = ak.local_index(sig_jets_all, axis=1)

pure_mask = (ak.fill_none(back_check, -1) == reco_idx) & (
    min_dr_reco < config["matching_cone_size"]
)

# Signal = all 4 jets pure; QCD = at least 1 impure
signal_mask_evt = ak.sum(pure_mask, axis=1) == 4
qcd_mask_evt = ~signal_mask_evt

n_signal = int(ak.sum(signal_mask_evt))
n_qcd = int(ak.sum(qcd_mask_evt))
n_total = int(ak.sum(has_4_tagged))
print(f"Signal events (all 4 pure): {n_signal}")
print(f"QCD background events (>=1 impure): {n_qcd}")
print(f"Total events with >=4 tagged jets: {n_total}")


# D_HH pairing
def pair_from_4jets(js4):
    """D_HH minimisation on 4-jet events."""
    j = [js4[:, i] for i in range(4)]
    perm_pairs = [([0, 1], [2, 3]), ([0, 2], [1, 3]), ([0, 3], [1, 2])]
    h_vecs = [
        (j[a].vector + j[b].vector, j[c].vector + j[d].vector)
        for (a, b), (c, d) in perm_pairs
    ]
    m1 = ak.concatenate([v[0].mass[:, None] for v in h_vecs], axis=1)
    m2 = ak.concatenate([v[1].mass[:, None] for v in h_vecs], axis=1)
    d_hh = abs(m1 - (125.0 / 120.0) * m2) / np.sqrt(1 + (125.0 / 120.0) ** 2)
    best = ak.argmin(d_hh, axis=1)

    c0, c1 = (best == 0), (best == 1)
    raw_h1 = ak.where(c0, h_vecs[0][0], ak.where(c1, h_vecs[1][0], h_vecs[2][0]))
    raw_h2 = ak.where(c0, h_vecs[0][1], ak.where(c1, h_vecs[1][1], h_vecs[2][1]))

    is_lead = raw_h1.pt >= raw_h2.pt
    lead = ak.where(is_lead, raw_h1, raw_h2)
    sub = ak.where(is_lead, raw_h2, raw_h1)
    hh = lead + sub
    return lead, sub, hh


# Signal reconstruction
sig_jets_4 = sig_jets_all[signal_mask_evt][:, :4]
if n_signal > 0:
    sig_lead, sig_sub, sig_hh = pair_from_4jets(sig_jets_4)
    print(
        f"\nSignal mHH: mean={ak.mean(sig_hh.mass):.1f}, median={np.median(ak.to_numpy(sig_hh.mass)):.1f} GeV"
    )
else:
    sig_lead = sig_sub = sig_hh = ak.Array([])
    print("\nNo signal events found!")

# QCD reconstruction
qcd_jets_4 = sig_jets_all[qcd_mask_evt][:, :4]
if n_qcd > 0:
    qcd_lead, qcd_sub, qcd_hh = pair_from_4jets(qcd_jets_4)
    print(
        f"QCD mHH: mean={ak.mean(qcd_hh.mass):.1f}, median={np.median(ak.to_numpy(qcd_hh.mass)):.1f} GeV"
    )
else:
    qcd_lead = qcd_sub = qcd_hh = ak.Array([])
    print("No QCD background events found!")


# --- Pairing Efficiency (Signal events only) ---
pair_eff = 0.0
eff_swapped = 0.0
if n_signal > 0:
    print("\n--- Computing Pairing Efficiency ---")
    # Get gen info for signal events only
    sig_gen_b = gen_b_for_match[signal_mask_evt]
    sig_gen_particles = dihiggs_events.GenPart[has_4_tagged][signal_mask_evt]

    # gmpr: for each reco jet, which gen b-quark is closest (index in gen_b array)
    dr_sig = sig_jets_4[:, :, None].vector.deltaR(sig_gen_b[:, None, :].vector)
    gmpr = ak.argmin(dr_sig, axis=2)  # (n_signal, 4)

    # Find true H1/H2 pairs from gen info
    def find_gen_b_pairs_with_indices(gmpr, gen_b_quarks, gen_particles):
        """
        Groups matched Gen b-quarks into pairs corresponding to their Higgs parents.
        Sorts pairs based on Parent Higgs pT (Leading vs Subleading).
        """
        matched_bs = gen_b_quarks[gmpr]
        parent_indices = matched_bs.genPartIdxMother
        sorter = ak.argsort(parent_indices, axis=1)
        sorted_bs = matched_bs[sorter]
        sorted_parents = parent_indices[sorter]
        pair_A_bs = sorted_bs[:, 0:2]
        pair_B_bs = sorted_bs[:, 2:4]
        parent_A_idx = sorted_parents[:, 0:1]
        parent_B_idx = sorted_parents[:, 2:3]
        parent_A = gen_particles[parent_A_idx][:, 0]
        parent_B = gen_particles[parent_B_idx][:, 0]
        is_A_leading = parent_A.pt > parent_B.pt
        h1_bs = ak.where(is_A_leading, pair_A_bs, pair_B_bs)
        h2_bs = ak.where(is_A_leading, pair_B_bs, pair_A_bs)
        sorted_gmpr = gmpr[sorter]
        pair_A_gmpr = sorted_gmpr[:, 0:2]
        pair_B_gmpr = sorted_gmpr[:, 2:4]
        h1_local_idxs = ak.where(is_A_leading, pair_A_gmpr, pair_B_gmpr)
        h2_local_idxs = ak.where(is_A_leading, pair_B_gmpr, pair_A_gmpr)
        return h1_bs, h2_bs, h1_local_idxs, h2_local_idxs

    _, _, true_h1_idxs, true_h2_idxs = find_gen_b_pairs_with_indices(
        gmpr, sig_gen_b, sig_gen_particles
    )
    true_h1_sorted = ak.sort(true_h1_idxs, axis=1)
    true_h2_sorted = ak.sort(true_h2_idxs, axis=1)

    # Reconstruct algo pairing with index tracking (same D_HH as pair_from_4jets)
    j = [sig_jets_4[:, i] for i in range(4)]
    perm_pairs = [([0, 1], [2, 3]), ([0, 2], [1, 3]), ([0, 3], [1, 2])]
    h_vecs = [
        (j[a].vector + j[b].vector, j[c].vector + j[d].vector)
        for (a, b), (c, d) in perm_pairs
    ]
    m1 = ak.concatenate([v[0].mass[:, None] for v in h_vecs], axis=1)
    m2 = ak.concatenate([v[1].mass[:, None] for v in h_vecs], axis=1)
    d_hh = abs(m1 - (125.0 / 120.0) * m2) / np.sqrt(1 + (125.0 / 120.0) ** 2)
    best = ak.argmin(d_hh, axis=1)

    # Map reco jet perm indices → gen b-quark indices via gmpr
    p1_c1 = gmpr[:, [0, 1]]
    p1_c2 = gmpr[:, [2, 3]]
    p2_c1 = gmpr[:, [0, 2]]
    p2_c2 = gmpr[:, [1, 3]]
    p3_c1 = gmpr[:, [0, 3]]
    p3_c2 = gmpr[:, [1, 2]]

    c0, c1_flag = (best == 0), (best == 1)
    algo_pair_A = ak.where(c0[:, None], p1_c1, ak.where(c1_flag[:, None], p2_c1, p3_c1))
    algo_pair_B = ak.where(c0[:, None], p1_c2, ak.where(c1_flag[:, None], p2_c2, p3_c2))

    # Apply pT ordering to algo pairs (same as pair_from_4jets)
    raw_h1_v = ak.where(c0, h_vecs[0][0], ak.where(c1_flag, h_vecs[1][0], h_vecs[2][0]))
    raw_h2_v = ak.where(c0, h_vecs[0][1], ak.where(c1_flag, h_vecs[1][1], h_vecs[2][1]))
    is_lead_v = raw_h1_v.pt >= raw_h2_v.pt
    algo_pair_leading = ak.where(is_lead_v[:, None], algo_pair_A, algo_pair_B)
    algo_pair_subleading = ak.where(is_lead_v[:, None], algo_pair_B, algo_pair_A)
    algo_A_sorted = ak.sort(algo_pair_leading, axis=1)
    algo_B_sorted = ak.sort(algo_pair_subleading, axis=1)

    # Check A: Algo leading matches Truth H1 AND Algo subleading matches Truth H2
    match_direct = ak.all(algo_A_sorted == true_h1_sorted, axis=1) & ak.all(
        algo_B_sorted == true_h2_sorted, axis=1
    )
    pair_eff = ak.mean(match_direct)
    print(f"Pairing Efficiency: {pair_eff:.2%}")

    # Check B: H1 and H2 are swapped
    match_swapped = ak.all(algo_A_sorted == true_h2_sorted, axis=1) & ak.all(
        algo_B_sorted == true_h1_sorted, axis=1
    )
    eff_swapped = ak.mean(match_swapped)
    print(f"Pairing Efficiency with pairs swapped: {eff_swapped:.2%}")
    print(f"Total Pairing Efficiency (including swapped): {pair_eff + eff_swapped:.2%}")
else:
    print("\nSkipping pairing efficiency (no signal events)")


# Store results
part_dihiggs_result = {
    "label": f"Trained ParT ({WP_SELECTION})",
    "n_total": n_total,
    "n_signal": n_signal,
    "n_qcd": n_qcd,
    "sig_lead": sig_lead,
    "sig_sub": sig_sub,
    "sig_hh": sig_hh,
    "qcd_lead": qcd_lead,
    "qcd_sub": qcd_sub,
    "qcd_hh": qcd_hh,
    "collection_key": collection_key,
    "wp": WP_SELECTION,
    "threshold": PART_BTAG_THRESHOLD,
    "has_regression": has_reg,
    "pair_eff": float(pair_eff),
    "eff_swapped": float(eff_swapped),
}

print(f"\n{'='*60}")
print(f"Di-Higgs reconstruction complete")
print(f"  Collection: {collection_key} ({dataset_used})")
print(f"  Working point: {WP_SELECTION} (threshold={PART_BTAG_THRESHOLD:.4f})")
print(
    f"  Selected: {n_total} events with >=4 tagged jets = {n_total/n_events_total*100:.2f}% of {n_events_total} total events"
)
print(
    f"  Selected: {len(ak.flatten(sig_jets_all))} jets {len(ak.flatten(sig_jets_all)) / len(ak.flatten(jets_tagged)) * 100:.2f}% of {len(ak.flatten(jets_tagged))} jets."
)
print(
    f"  Signal: {n_signal} events ({n_signal/n_total*100:.1f}%) | QCD: {n_qcd} events ({n_qcd/n_total*100:.1f}%)"
)
print(f"  pT regression: {'applied' if has_reg else 'not available'}")
print(
    f"  Pairing eff: {pair_eff:.2%} | Swapped: {eff_swapped:.2%} | Total: {pair_eff + eff_swapped:.2%}"
)
print(f"{'='*60}")

In [ ]:
# ============================================================
# TOP-N JET PURITY EFFICIENCY: pT ordering vs ParT b-tag ordering
# ============================================================
from analysis_helpers import get_pure_jet_idxs_cross_matched


def get_eff_first_jet_pure(gen_b_quarks, reco_jets, tagger_name, n, k):
    """
    Calculates the efficiency of finding at least n pure jets in the top k jets
    ordered by pT and by the specified b-tagging score.

    gen_b_quarks: generated b-quarks
    reco_jets: reconstructed jets
    tagger_name: name of the b-tagging score to use for ordering
    n: Number of pure jets to find
    k: Max number of top jets to consider
    -----------
    Returns:
    eff_highest_pt_pure: Efficiency of highest pT jet being pure
    eff_highest_tag_pure: Efficiency of highest b-tagging score jet being pure
    more_than_n_eff_pt: List of efficiencies of finding at least n pure jets in top k jets ordered by pT
    more_than_n_eff_tag: List of efficiencies of finding at least n pure jets in top k jets ordered by b-tagging score
    -----------
    """
    pt_ordered = reco_jets[ak.argsort(reco_jets.vector.pt, ascending=False)]
    tag_ordered = reco_jets[ak.argsort(reco_jets[tagger_name], ascending=False)]

    pt_ordered_purity_idxs = get_pure_jet_idxs_cross_matched(gen_b_quarks, pt_ordered)
    tag_ordered_purity_idxs = get_pure_jet_idxs_cross_matched(gen_b_quarks, tag_ordered)

    # Eff highest pt jet is pure
    num_highest_pt_pure = ak.any(pt_ordered_purity_idxs == 0, axis=1)
    eff_highest_pt_pure = ak.sum(num_highest_pt_pure) / len(gen_b_quarks)
    print(f"Eff highest pt jet is pure: {eff_highest_pt_pure:.4f}")

    # Eff highest btag jet is pure
    num_highest_tag_pure = ak.any(tag_ordered_purity_idxs == 0, axis=1)
    eff_highest_tag_pure = ak.sum(num_highest_tag_pure) / len(gen_b_quarks)
    print(f"Eff highest {tagger_name} jet is pure: {eff_highest_tag_pure:.4f}")

    # Eff finding n pure jets in top k jets considered
    n = 4
    range_k = range(k + 1)

    more_than_n_eff_pt = []
    more_than_n_eff_tag = []
    for k in range_k:
        num_k_highest_pt_pure = (ak.num(pt_ordered_purity_idxs) == n) & (
            ak.all(pt_ordered_purity_idxs < k, axis=1)
        )
        eff_k_highest_pt_pure = ak.sum(num_k_highest_pt_pure) / len(gen_b_quarks)
        more_than_n_eff_pt.append(eff_k_highest_pt_pure)

        num_k_highest_tag_pure = (ak.num(tag_ordered_purity_idxs) == n) & (
            ak.all(tag_ordered_purity_idxs < k, axis=1)
        )
        eff_k_highest_tag_pure = ak.sum(num_k_highest_tag_pure) / len(gen_b_quarks)
        more_than_n_eff_tag.append(eff_k_highest_tag_pure)

    return (
        eff_highest_pt_pure,
        eff_highest_tag_pure,
        more_than_n_eff_pt,
        more_than_n_eff_tag,
    )


# --- Run efficiency analysis ---
n = 4  # Number of b-jets to find
k = 25  # Max number of top jets to consider
range_k = range(k + 1)

(
    eff_highest_pt_pure_part,
    eff_highest_btag_pure_part,
    more_than_n_eff_pt_part,
    more_than_n_eff_btag_part,
) = get_eff_first_jet_pure(dihiggs_gen_b, scored_jets, "btag_score", n, k)

# --- Plot 1: pT ordering ---
print(f"\nPlotting Top-N Jet Efficiencies for N = {n}...")
fig_eff_pt, ax_pt = plt.subplots(figsize=(10, 6))
ax_pt.step(
    range_k, more_than_n_eff_pt_part, where="mid", label="ParT pT", color="purple"
)
ax_pt.set_xlabel("k (Number of Top Jets Considered)")
ax_pt.set_ylabel(f"Probability of Finding at least {n} b-jets")
ax_pt.set_title(
    f"Probability of Finding at least {n} b-jets vs. Top k Jets (pT ordered)"
)
ax_pt.grid(True, linestyle="--", alpha=0.6)
ax_pt.legend()
save_fig(fig_eff_pt, "top_n_eff_pt_ordering")
plt.show()

# --- Plot 2: b-tag ordering ---
fig_eff_btag, ax_btag = plt.subplots(figsize=(10, 6))
ax_btag.step(
    range_k, more_than_n_eff_btag_part, where="mid", label="ParT BTag", color="purple"
)
ax_btag.set_xlabel("k (Number of Top Jets Considered)")
ax_btag.set_ylabel(f"Probability of Finding at least {n} b-jets")
ax_btag.set_title(
    f"Probability of Finding at least {n} b-jets vs. Top k Jets (BTag ordered)"
)
ax_btag.grid(True, linestyle="--", alpha=0.6)
ax_btag.legend()
save_fig(fig_eff_btag, "top_n_eff_btag_ordering")
plt.show()

In [ ]:
# ============================================================
# DI-HIGGS MASS DISTRIBUTION PLOTS + SAVE
# ============================================================
from matplotlib.patches import Ellipse

res = part_dihiggs_result
label = res["label"]
color = "purple"

sig_window_h = (90, 160)
sig_window_hh = (250, 550)

# --- 1. Signal vs QCD: 1×3 overlay (leading mH, subleading mH, mHH) ---
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
bins_h = np.linspace(0, 300, 61)
bins_hh = np.linspace(200, 800, 61)

# Leading Higgs
ax = axes[0]
if res["n_signal"] > 0:
    ax.hist(
        ak.to_numpy(res["sig_lead"].mass),
        bins=bins_h,
        histtype="stepfilled",
        alpha=0.3,
        color=color,
        label=f'Signal ({res["n_signal"]})',
        density=True,
    )
    ax.hist(
        ak.to_numpy(res["sig_lead"].mass),
        bins=bins_h,
        histtype="step",
        linewidth=2,
        color=color,
        density=True,
    )
if res["n_qcd"] > 0:
    ax.hist(
        ak.to_numpy(res["qcd_lead"].mass),
        bins=bins_h,
        histtype="step",
        linewidth=2,
        color="grey",
        linestyle="--",
        label=f'QCD bkg ({res["n_qcd"]})',
        density=True,
    )
ax.axvline(125, color="green", linestyle=":", linewidth=1.5)
ax.axvspan(*sig_window_h, alpha=0.05, color="green")
ax.set_xlabel("Leading $m_H$ [GeV]")
ax.set_ylabel("Events / 5 GeV")
ax.set_title(f"{label} — Leading Higgs")
ax.legend(fontsize=10)

# Subleading Higgs
ax = axes[1]
if res["n_signal"] > 0:
    ax.hist(
        ak.to_numpy(res["sig_sub"].mass),
        bins=bins_h,
        histtype="stepfilled",
        alpha=0.3,
        color=color,
        label="Signal",
        density=True,
    )
    ax.hist(
        ak.to_numpy(res["sig_sub"].mass),
        bins=bins_h,
        histtype="step",
        linewidth=2,
        color=color,
        density=True,
    )
if res["n_qcd"] > 0:
    ax.hist(
        ak.to_numpy(res["qcd_sub"].mass),
        bins=bins_h,
        histtype="step",
        linewidth=2,
        color="grey",
        linestyle="--",
        label="QCD bkg",
        density=True,
    )
ax.axvline(125, color="green", linestyle=":", linewidth=1.5)
ax.axvspan(*sig_window_h, alpha=0.05, color="green")
ax.set_xlabel("Subleading $m_H$ [GeV]")
ax.set_ylabel("Events / 5 GeV")
ax.set_title(f"{label} — Subleading Higgs")
ax.legend(fontsize=10)

# mHH
ax = axes[2]
if res["n_signal"] > 0:
    ax.hist(
        ak.to_numpy(res["sig_hh"].mass),
        bins=bins_hh,
        histtype="stepfilled",
        alpha=0.3,
        color=color,
        label="Signal",
        density=True,
    )
    ax.hist(
        ak.to_numpy(res["sig_hh"].mass),
        bins=bins_hh,
        histtype="step",
        linewidth=2,
        color=color,
        density=True,
    )
if res["n_qcd"] > 0:
    ax.hist(
        ak.to_numpy(res["qcd_hh"].mass),
        bins=bins_hh,
        histtype="step",
        linewidth=2,
        color="grey",
        linestyle="--",
        label="QCD bkg",
        density=True,
    )
ax.axvspan(*sig_window_hh, alpha=0.05, color="green")
ax.set_xlabel("$m_{HH}$ [GeV]")
ax.set_ylabel("Events / 10 GeV")
ax.set_title(f"{label} — $m_{{HH}}$")
ax.legend(fontsize=10)

reg_tag = (
    "pT-corrected"
    if res["has_regression"] and apply_pt_correction
    else "uncorrected pT"
)
fig.suptitle(f"Di-Higgs Reconstruction — {label} ({reg_tag})", fontsize=16, y=1.01)
plt.tight_layout()
save_fig(fig, f"dihiggs_mass_1d_{WP_SELECTION}")
plt.show()


# --- 2. 2D mH1 vs mH2 (Signal vs QCD) ---
def R_hh_func(mh1, mh2):
    return np.sqrt((mh1 - 125.0) ** 2 + (mh2 - 120.0) ** 2)


bins_2d = np.linspace(0, 300, 61)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax_idx, (category, lead_key, sub_key, n_events) in enumerate(
    [
        ("Signal", "sig_lead", "sig_sub", res["n_signal"]),
        ("QCD Background", "qcd_lead", "qcd_sub", res["n_qcd"]),
    ]
):
    ax = axes[ax_idx]
    if n_events > 0:
        lead_mass = ak.to_numpy(res[lead_key].mass)
        sub_mass = ak.to_numpy(res[sub_key].mass)
        r_hh_vals = R_hh_func(lead_mass, sub_mass)
        sel = r_hh_vals < 550.0

        h = ax.hist2d(
            lead_mass[sel], sub_mass[sel], bins=[bins_2d, bins_2d], cmap="viridis"
        )
        ax.axvline(
            125, color="red", linestyle="--", linewidth=1.5, label="$m_H$ = 125 GeV"
        )
        ax.axhline(120, color="red", linestyle="--", linewidth=1.5)
        ellipse = Ellipse(
            xy=(125, 120),
            width=110,
            height=96,
            angle=0,
            edgecolor="yellow",
            facecolor="none",
            linestyle="--",
            linewidth=2,
            label="$R_{HH}$ = 55 GeV",
        )
        ax.add_patch(ellipse)
        n_sel = int(np.sum(sel))
        ax.set_title(
            f"{label} — {category}\n({n_sel} events inside $R_{{HH}}$ / {n_events} total)"
        )
        fig.colorbar(h[3], ax=ax, label="Events")
    else:
        ax.text(
            0.5,
            0.5,
            f"No {category.lower()} events",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=14,
        )
        ax.set_title(f"{label} — {category}")
    ax.set_xlabel("Leading Higgs Mass [GeV]")
    ax.set_ylabel("Subleading Higgs Mass [GeV]")
    ax.legend(loc="upper right", fontsize=10)

fig.suptitle(f"2D $m_{{H1}}$ vs $m_{{H2}}$ — {label} ({reg_tag})", fontsize=16, y=1.02)
plt.tight_layout()
save_fig(fig, f"dihiggs_mass_2d_{WP_SELECTION}")
plt.show()

# --- 3. Significance summary ---
print(f"\n{'='*70}")
print(
    f"{'Tagger':<30} {'S':>6} {'B':>7} {'S/√(S+B)':>10}  |  {'S (window)':>10} {'B (window)':>10} {'S/√(S+B)':>10}"
)
print(
    f"{'':30} {'(all)':>6} {'(all)':>7} {'(all)':>10}  |  {str(sig_window_hh):>10} {str(sig_window_hh):>10} {'(window)':>10}"
)
print("-" * 70)
if res["n_signal"] > 0 and res["n_qcd"] > 0:
    sig_m = ak.to_numpy(res["sig_hh"].mass)
    bkg_m = ak.to_numpy(res["qcd_hh"].mass)
    S_all, B_all = len(sig_m), len(bkg_m)
    signif_all = S_all / np.sqrt(S_all + B_all) if (S_all + B_all) > 0 else 0
    S_win = np.sum((sig_m >= sig_window_hh[0]) & (sig_m <= sig_window_hh[1]))
    B_win = np.sum((bkg_m >= sig_window_hh[0]) & (bkg_m <= sig_window_hh[1]))
    signif_win = S_win / np.sqrt(S_win + B_win) if (S_win + B_win) > 0 else 0
    print(
        f"{label:<30} {S_all:>6} {B_all:>7} {signif_all:>10.2f}  |  {S_win:>10} {B_win:>10} {signif_win:>10.2f}"
    )
else:
    print(f"{label:<30}  Insufficient events for significance")
print("=" * 70)
print(f"\nWorking point: {WP_SELECTION} (threshold={PART_BTAG_THRESHOLD:.4f})")
print(
    f"pT regression: {'applied' if res['has_regression'] and apply_pt_correction else 'not applied'}"
)
print(f"Collection: {res['collection_key']}")

print(f"\nAll di-Higgs plots saved to: {plot_dir}")

In [ ]:
# ============================================================
# ATTENTION MAP VISUALIZATION & PAIRWISE FEATURE ANALYSIS
# ============================================================
import torch
import torch.nn as nn
from parT import to_rapidity, to_energy


# ============================================================
# 1. EXTRACT PAIRWISE FEATURES FOR VISUALIZATION
# ============================================================
def compute_pairwise_features(x, mask=None):
    """
    Compute the raw pairwise features before they go through the MLP.
    Returns: delta, k_t, z, m_2, d_dxy, d_z0, q_ij (all in log-space where applicable)
    """
    B, N, C = x.shape

    if mask is None:
        mask = x[..., 0] != 0

    # Get physical quantities
    r = to_rapidity(x).unsqueeze(2)  # rapidity
    phi = x[..., 3].unsqueeze(2)
    pt = x[..., 1].unsqueeze(2)

    # Delta R (angular distance)
    delta = torch.sqrt(
        (r - r.transpose(1, 2)) ** 2 + (phi - phi.transpose(1, 2)) ** 2 + 1e-8
    )

    # k_T (relative transverse momentum)
    k_t = torch.min(pt, pt.transpose(1, 2)) * delta

    # z (momentum fraction)
    z = torch.min(pt, pt.transpose(1, 2)) / (pt + pt.transpose(1, 2) + 1e-8)

    # m^2 (invariant mass squared)
    m_2 = (
        2
        * pt
        * pt.transpose(1, 2)
        * (torch.cosh(r - r.transpose(1, 2)) - torch.cos(phi - phi.transpose(1, 2)))
        + 1e-8
    )

    # Impact parameter differences
    dxy = x[..., 4].unsqueeze(2)
    z0 = x[..., 5].unsqueeze(2)
    d_dxy = dxy - dxy.transpose(1, 2)
    d_z0 = z0 - z0.transpose(1, 2)

    # Charge product
    q = x[..., 6].unsqueeze(2)
    pt_jet = pt.sum(dim=1, keepdim=True)
    q_weighted = q * pt / (pt_jet + 1e-8)
    q_ij = q_weighted * q_weighted.transpose(1, 2)

    # Apply log transform (as done in the model)
    delta_log = torch.log(torch.clamp(delta, min=1e-8))
    k_t_log = torch.log(torch.clamp(k_t, min=1e-8))
    z_log = torch.log(torch.clamp(z, min=1e-8))
    m_2_log = torch.log(torch.clamp(m_2, min=1e-8))

    return {
        "delta_R": delta,
        "log_delta_R": delta_log,
        "k_t": k_t,
        "log_k_t": k_t_log,
        "z": z,
        "log_z": z_log,
        "m_2": m_2,
        "log_m_2": m_2_log,
        "d_dxy": d_dxy,
        "d_z0": d_z0,
        "q_ij": q_ij,
    }, mask


# ============================================================
# 2. ATTENTION HOOK CLASS
# ============================================================
class AttentionHook:
    """Hook to capture attention weights from ParticleAttentionBlock."""

    def __init__(self):
        self.attention_weights = []
        self.handles = []

    def hook_fn(self, module, input, output):
        # We need to modify the forward to capture attention
        # This hook captures the output, but we need the attention weights
        pass

    def clear(self):
        self.attention_weights = []
        for h in self.handles:
            h.remove()
        self.handles = []


# Modified forward that returns attention weights
def forward_with_attention(model, x, particle_mask=None):
    """
    Forward pass that also returns attention weights from all layers.
    """
    model.eval()
    attention_maps = {"particle_attn": [], "class_attn": []}

    B = x.shape[0]
    x_raw = x.clone()

    # Preprocessing
    if model.use_batch_norm:
        x_proc = x.transpose(1, 2)
        x_proc = model.input_bn(x_proc)
        x_proc = x_proc.transpose(1, 2)
    else:
        x_proc = model.input_ln(x)

    # Compute pairwise features
    u_ij = model.pair_embed(x_raw, particle_mask)

    # Embedding
    x_embed = model.embed(x_proc)

    # Particle attention blocks
    x_curr = x_embed
    for block in model.part_atten_blocks:
        # Extract attention weights manually
        B_curr, N, C = x_curr.shape

        residual = x_curr
        x_norm = block.norm1(x_curr)

        qkv = block.qkv(x_norm).reshape(
            B_curr, N, 3, block.num_heads, C // block.num_heads
        )
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * block.scale

        if particle_mask is not None:
            attn_mask = particle_mask.unsqueeze(1).unsqueeze(2)
            attn = attn.masked_fill(~attn_mask, float("-inf"))

        if u_ij is not None:
            pair_bias = block.pair_proj(u_ij).permute(0, 3, 1, 2)
            attn = attn + pair_bias

        attn_weights = attn.softmax(dim=-1)
        attention_maps["particle_attn"].append(attn_weights.detach().cpu())

        # Continue forward pass
        x_curr = block(x_curr, u_ij, particle_mask)

    # Add CLS token
    cls_tokens = model.cls_token.expand(B, -1, -1)
    x_with_cls = torch.cat((cls_tokens, x_curr), dim=1)

    # Class attention blocks
    for block in model.cls_atten_blocks:
        B_curr, N_plus_1, C = x_with_cls.shape

        residual = x_with_cls
        x_norm = block.norm1(x_with_cls)

        qkv = block.qkv(x_norm).reshape(
            B_curr, N_plus_1, 3, block.num_heads, C // block.num_heads
        )
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        q_cls = q[:, :, 0:1, :]
        attn = (q_cls @ k.transpose(-2, -1)) * block.scale

        if particle_mask is not None:
            attn_mask_cls = torch.cat(
                [
                    torch.ones(
                        (B_curr, 1, 1, 1),
                        dtype=particle_mask.dtype,
                        device=particle_mask.device,
                    ),
                    particle_mask.unsqueeze(1).unsqueeze(2),
                ],
                dim=-1,
            )
            attn = attn.masked_fill(~attn_mask_cls, float("-inf"))

        attn_weights = attn.softmax(dim=-1)
        attention_maps["class_attn"].append(attn_weights.detach().cpu())

        x_with_cls = block(x_with_cls, particle_mask)

    return attention_maps, u_ij.detach().cpu()


# ============================================================
# 3. COMPUTE AND VISUALIZE
# ============================================================
print("=" * 60)
print("ATTENTION & PAIRWISE FEATURE ANALYSIS")
print("=" * 60)

# Select sample jets (signal and background)
n_samples = 5
signal_indices = np.where(all_labels == 1)[0][:n_samples]
background_indices = np.where(all_labels == 0)[0][:n_samples]
sample_indices = np.concatenate([signal_indices, background_indices])

# Get sample data
sample_x = all_constituents[sample_indices].to(device)
sample_mask = sample_x[:, :, 1] > 0  # pT > 0 means valid constituent

# Compute pairwise features
pairwise_feats, mask = compute_pairwise_features(sample_x.cpu(), sample_mask.cpu())

# Get attention maps
model.eval()
with torch.no_grad():
    attention_maps, u_ij = forward_with_attention(model, sample_x, sample_mask)

print(
    f"Analyzed {len(sample_indices)} jets ({n_samples} signal, {n_samples} background)"
)
print(f"Particle attention layers: {len(attention_maps['particle_attn'])}")
print(f"Class attention layers: {len(attention_maps['class_attn'])}")

# ============================================================
# 4. PLOT PAIRWISE FEATURE DISTRIBUTIONS
# ============================================================
# Use full dataset for distribution plots
print("\nComputing pairwise features on full validation set...")
n_jets_for_dist = min(1000, len(all_constituents))
dist_x = all_constituents[:n_jets_for_dist]
dist_mask = dist_x[:, :, 1] > 0
dist_labels = all_labels[:n_jets_for_dist]

pairwise_dist, _ = compute_pairwise_features(dist_x, dist_mask)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

feature_names_pairwise = [
    ("log_delta_R", r"$\log(\Delta R)$"),
    ("log_k_t", r"$\log(k_T)$"),
    ("log_z", r"$\log(z)$"),
    ("log_m_2", r"$\log(m^2)$"),
    ("d_dxy", r"$\Delta d_{xy}$"),
    ("d_z0", r"$\Delta z_0$"),
    ("q_ij", r"$q_i \cdot q_j$"),
]

for idx, (feat_key, feat_label) in enumerate(feature_names_pairwise):
    ax = axes[idx]

    feat = pairwise_dist[feat_key].numpy()
    mask_np = dist_mask.numpy()

    # Get upper triangular values (avoid double counting i,j and j,i)
    sig_vals = []
    bkg_vals = []

    for i in range(len(feat)):
        m = mask_np[i]
        n_valid = m.sum()
        if n_valid > 1:
            f = feat[i, :n_valid, :n_valid]
            upper_tri = f[np.triu_indices(n_valid, k=1)]
            if dist_labels[i] == 1:
                sig_vals.extend(upper_tri)
            else:
                bkg_vals.extend(upper_tri)

    sig_vals = np.array(sig_vals)
    bkg_vals = np.array(bkg_vals)

    # Filter out inf/nan
    sig_vals = sig_vals[np.isfinite(sig_vals)]
    bkg_vals = bkg_vals[np.isfinite(bkg_vals)]

    if len(sig_vals) > 0 and len(bkg_vals) > 0:
        range_min = min(np.percentile(sig_vals, 1), np.percentile(bkg_vals, 1))
        range_max = max(np.percentile(sig_vals, 99), np.percentile(bkg_vals, 99))

        ax.hist(
            sig_vals,
            bins=50,
            range=(range_min, range_max),
            histtype="step",
            label="Signal",
            color="blue",
            density=True,
        )
        ax.hist(
            bkg_vals,
            bins=50,
            range=(range_min, range_max),
            histtype="step",
            label="Background",
            color="red",
            density=True,
        )

    ax.set_xlabel(feat_label)
    ax.set_ylabel("Density")
    ax.set_title(f"Pairwise Feature: {feat_label}")
    ax.legend()

# Hide unused subplot
axes[-1].axis("off")

plt.tight_layout()
save_fig(fig, "pairwise_feature_distributions")
plt.show()


# ============================================================
# 5. ATTENTION MAP VISUALIZATION
# ============================================================
def plot_attention_maps(
    attention_maps, sample_idx, sample_mask, is_signal, layer_idx=0
):
    """Plot attention maps for a single jet."""
    n_heads = attention_maps["particle_attn"][layer_idx].shape[1]
    n_valid = sample_mask[sample_idx].sum().item()

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    attn = attention_maps["particle_attn"][layer_idx][sample_idx, :, :n_valid, :n_valid]

    for h in range(min(n_heads, 8)):
        ax = axes[h]
        attn_head = attn[h].numpy()

        im = ax.imshow(
            attn_head, cmap="viridis", aspect="auto", vmin=0, vmax=attn_head.max()
        )
        ax.set_xlabel("Key Particle")
        ax.set_ylabel("Query Particle")
        ax.set_title(f"Head {h+1}")
        plt.colorbar(im, ax=ax, fraction=0.046)

    label = "Signal (b-jet)" if is_signal else "Background"
    fig.suptitle(
        f"Particle Attention Maps - Layer {layer_idx+1} - {label}", fontsize=14
    )
    plt.tight_layout()
    return fig


# Plot attention maps for signal and background examples
print("\nPlotting attention maps...")

# Signal example
fig_sig = plot_attention_maps(
    attention_maps, 0, sample_mask.cpu(), is_signal=True, layer_idx=0
)
save_fig(fig_sig, "attention_map_signal_layer1")
plt.show()

# Background example
fig_bkg = plot_attention_maps(
    attention_maps, n_samples, sample_mask.cpu(), is_signal=False, layer_idx=0
)
save_fig(fig_bkg, "attention_map_background_layer1")
plt.show()

# ============================================================
# 6. CLASS ATTENTION VISUALIZATION
# ============================================================
fig, axes = plt.subplots(2, n_samples, figsize=(4 * n_samples, 8))

for i in range(n_samples):
    # Signal
    cls_attn = (
        attention_maps["class_attn"][-1][i, :, 0, 1:].mean(dim=0).numpy()
    )  # Average over heads
    n_valid = sample_mask[i].sum().item()

    ax = axes[0, i]
    ax.bar(range(n_valid), cls_attn[:n_valid])
    ax.set_xlabel("Constituent Index")
    ax.set_ylabel("Attention Weight")
    ax.set_title(f"Signal Jet {i+1}")

    # Background
    cls_attn = (
        attention_maps["class_attn"][-1][n_samples + i, :, 0, 1:].mean(dim=0).numpy()
    )
    n_valid = sample_mask[n_samples + i].sum().item()

    ax = axes[1, i]
    ax.bar(range(n_valid), cls_attn[:n_valid], color="red")
    ax.set_xlabel("Constituent Index")
    ax.set_ylabel("Attention Weight")
    ax.set_title(f"Background Jet {i+1}")

fig.suptitle(
    "Class Token Attention Weights (Which particles the classifier focuses on)",
    fontsize=14,
)
plt.tight_layout()
save_fig(fig, "class_attention_weights")
plt.show()

# ============================================================
# 7. AVERAGE ATTENTION PATTERNS
# ============================================================
# Compute average attention across more samples
n_avg = min(100, len(all_constituents))
avg_x = all_constituents[:n_avg].to(device)
avg_mask = avg_x[:, :, 1] > 0
avg_labels = all_labels[:n_avg]

with torch.no_grad():
    avg_attention_maps, _ = forward_with_attention(model, avg_x, avg_mask)

# Compute average attention as a function of Delta R
print("\nComputing attention vs Delta R correlation...")
pairwise_avg, _ = compute_pairwise_features(avg_x.cpu(), avg_mask.cpu())

delta_r_vals = []
attn_vals = []
labels_vals = []

for i in range(n_avg):
    n_valid = avg_mask[i].sum().item()
    if n_valid > 1:
        dr = pairwise_avg["delta_R"][i, :n_valid, :n_valid].numpy()
        attn = (
            avg_attention_maps["particle_attn"][0][i, :, :n_valid, :n_valid]
            .mean(dim=0)
            .numpy()
        )

        for j in range(n_valid):
            for k in range(j + 1, n_valid):
                delta_r_vals.append(dr[j, k])
                attn_vals.append(attn[j, k])
                labels_vals.append(avg_labels[i])

delta_r_vals = np.array(delta_r_vals)
attn_vals = np.array(attn_vals)
labels_vals = np.array(labels_vals)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hexbin plot
ax = axes[0]
hb = ax.hexbin(
    delta_r_vals,
    attn_vals,
    gridsize=50,
    cmap="viridis",
    extent=[0, 2, 0, np.percentile(attn_vals, 99)],
    mincnt=1,
)
ax.set_xlabel(r"$\Delta R$")
ax.set_ylabel("Attention Weight")
ax.set_title("Attention Weight vs Angular Distance")
plt.colorbar(hb, ax=ax, label="Count")

# Separate by signal/background
ax = axes[1]
sig_mask_v = labels_vals == 1
dr_bins = np.linspace(0, 2, 20)
sig_attn_means = []
bkg_attn_means = []

for j in range(len(dr_bins) - 1):
    bin_mask = (delta_r_vals >= dr_bins[j]) & (delta_r_vals < dr_bins[j + 1])
    sig_attn_means.append(
        np.mean(attn_vals[bin_mask & sig_mask_v])
        if (bin_mask & sig_mask_v).sum() > 0
        else np.nan
    )
    bkg_attn_means.append(
        np.mean(attn_vals[bin_mask & ~sig_mask_v])
        if (bin_mask & ~sig_mask_v).sum() > 0
        else np.nan
    )

dr_centers = (dr_bins[:-1] + dr_bins[1:]) / 2
ax.plot(dr_centers, sig_attn_means, "b-o", label="Signal", markersize=5)
ax.plot(dr_centers, bkg_attn_means, "r-o", label="Background", markersize=5)
ax.set_xlabel(r"$\Delta R$")
ax.set_ylabel("Mean Attention Weight")
ax.set_title("Attention vs Distance by Jet Type")
ax.legend()

plt.tight_layout()
save_fig(fig, "attention_vs_delta_r")
plt.show()

print(f"\n{'='*60}")
print("Attention and pairwise feature analysis complete!")
print(f"{'='*60}")

In [ ]:
# ============================================================
# LAYER ACTIVATION VISUALIZATION
# ============================================================
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import torch.nn.functional as F


# ============================================================
# 1. FORWARD WITH LAYER ACTIVATIONS
# ============================================================
def forward_with_activations(model, x, particle_mask=None):
    """
    Forward pass that captures intermediate layer activations.
    Returns activations at each stage of the network.
    """
    model.eval()
    activations = {
        "input": x.detach().cpu(),
        "input_processed": None,
        "embedding": None,
        "pairwise_embed": None,
        "particle_attn_layers": [],
        "pre_cls": None,
        "cls_attn_layers": [],
        "final_cls_token": None,
        "output_logits": None,
    }

    B = x.shape[0]
    x_raw = x.clone()

    # Input preprocessing
    if model.use_batch_norm:
        x_proc = x.transpose(1, 2)
        x_proc = model.input_bn(x_proc)
        x_proc = x_proc.transpose(1, 2)
    else:
        x_proc = model.input_ln(x)

    activations["input_processed"] = x_proc.detach().cpu()

    # Pairwise embedding
    u_ij = model.pair_embed(x_raw, particle_mask)
    activations["pairwise_embed"] = u_ij.detach().cpu()

    # Linear embedding
    x_embed = model.embed(x_proc)
    activations["embedding"] = x_embed.detach().cpu()

    # Particle attention blocks
    x_curr = x_embed
    for i, block in enumerate(model.part_atten_blocks):
        x_curr = block(x_curr, u_ij, particle_mask)
        activations["particle_attn_layers"].append(x_curr.detach().cpu())

    # Add CLS token
    cls_tokens = model.cls_token.expand(B, -1, -1)
    x_with_cls = torch.cat((cls_tokens, x_curr), dim=1)
    activations["pre_cls"] = x_with_cls.detach().cpu()

    # Class attention blocks
    for i, block in enumerate(model.cls_atten_blocks):
        x_with_cls = block(x_with_cls, particle_mask)
        activations["cls_attn_layers"].append(x_with_cls.detach().cpu())

    # Final CLS token
    cls_token_final = x_with_cls[:, 0]
    activations["final_cls_token"] = cls_token_final.detach().cpu()

    # Output
    logits = model.head(cls_token_final)
    activations["output_logits"] = logits.detach().cpu()

    return activations


# ============================================================
# 2. COLLECT ACTIVATIONS
# ============================================================
print("=" * 60)
print("LAYER ACTIVATION ANALYSIS")
print("=" * 60)

# Use a subset of validation data for visualization
n_vis = min(500, len(all_constituents))
vis_x = all_constituents[:n_vis].to(device)
vis_mask = vis_x[:, :, 1] > 0
vis_labels = all_labels[:n_vis]

print(f"Analyzing activations for {n_vis} jets...")
print(f"  Signal jets: {(vis_labels == 1).sum()}")
print(f"  Background jets: {(vis_labels == 0).sum()}")

with torch.no_grad():
    activations = forward_with_activations(model, vis_x, vis_mask)

# ============================================================
# 3. ACTIVATION MAGNITUDE DISTRIBUTIONS
# ============================================================
print("\nPlotting activation magnitude distributions...")

layer_names = (
    ["embedding"]
    + [f"particle_attn_{i+1}" for i in range(len(activations["particle_attn_layers"]))]
    + ["pre_cls"]
    + [f"cls_attn_{i+1}" for i in range(len(activations["cls_attn_layers"]))]
    + ["final_cls_token"]
)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

# Get activations for each layer
layer_activations = [activations["embedding"]]
layer_activations.extend(activations["particle_attn_layers"])
layer_activations.append(activations["pre_cls"])
layer_activations.extend(activations["cls_attn_layers"])
layer_activations.append(activations["final_cls_token"].unsqueeze(1))

for idx, (name, act) in enumerate(zip(layer_names, layer_activations)):
    if idx >= len(axes):
        break

    ax = axes[idx]

    # Flatten activations
    if len(act.shape) == 3:
        # For sequence activations, average over sequence dim
        act_flat = act.mean(dim=1).numpy()
    else:
        act_flat = act.numpy()

    # Plot distribution of activation magnitudes
    sig_act = act_flat[vis_labels == 1].flatten()
    bkg_act = act_flat[vis_labels == 0].flatten()

    range_min = min(np.percentile(sig_act, 1), np.percentile(bkg_act, 1))
    range_max = max(np.percentile(sig_act, 99), np.percentile(bkg_act, 99))

    ax.hist(
        sig_act,
        bins=50,
        range=(range_min, range_max),
        histtype="step",
        label="Signal",
        color="blue",
        density=True,
        alpha=0.8,
    )
    ax.hist(
        bkg_act,
        bins=50,
        range=(range_min, range_max),
        histtype="step",
        label="Background",
        color="red",
        density=True,
        alpha=0.8,
    )

    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Density")
    ax.set_title(f"{name}\n(dim={act_flat.shape[-1]})")
    ax.legend(fontsize=8)

# Hide unused subplots
for idx in range(len(layer_names), len(axes)):
    axes[idx].axis("off")

plt.suptitle("Activation Magnitude Distributions Across Layers", fontsize=14)
plt.tight_layout()
save_fig(fig, "activation_magnitude_distributions")
plt.show()

# ============================================================
# 4. t-SNE VISUALIZATION OF LAYER REPRESENTATIONS
# ============================================================
print("\nComputing t-SNE projections...")

# Select key layers for t-SNE
tsne_layers = {
    "Input (processed)": activations["input_processed"].mean(dim=1).numpy(),
    "After Embedding": activations["embedding"].mean(dim=1).numpy(),
    f"After ParticleAttn-{len(activations['particle_attn_layers'])}": activations[
        "particle_attn_layers"
    ][-1]
    .mean(dim=1)
    .numpy(),
    "Final CLS Token": activations["final_cls_token"].numpy(),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (layer_name, layer_act) in enumerate(tsne_layers.items()):
    ax = axes[idx]

    # Apply PCA first to reduce dimensionality if needed
    if layer_act.shape[1] > 50:
        pca = PCA(n_components=50, random_state=42)
        layer_act_reduced = pca.fit_transform(layer_act)
    else:
        layer_act_reduced = layer_act

    # t-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, n_vis // 4))
    tsne_result = tsne.fit_transform(layer_act_reduced)

    # Plot
    ax.scatter(
        tsne_result[vis_labels == 0, 0],
        tsne_result[vis_labels == 0, 1],
        c="red",
        alpha=0.5,
        s=10,
        label="Background",
    )
    ax.scatter(
        tsne_result[vis_labels == 1, 0],
        tsne_result[vis_labels == 1, 1],
        c="blue",
        alpha=0.5,
        s=10,
        label="Signal",
    )

    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.set_title(layer_name)
    ax.legend()

plt.suptitle(
    "t-SNE Visualization: How Representations Evolve Through Layers", fontsize=14
)
plt.tight_layout()
save_fig(fig, "tsne_layer_evolution")
plt.show()

# ============================================================
# 5. NEURON ACTIVATION PATTERNS
# ============================================================
print("\nAnalyzing neuron activation patterns...")

# Focus on final CLS token activations
final_act = activations["final_cls_token"].numpy()
embed_dim = final_act.shape[1]

# Compute mean activation per neuron for signal vs background
sig_neuron_mean = final_act[vis_labels == 1].mean(axis=0)
bkg_neuron_mean = final_act[vis_labels == 0].mean(axis=0)
neuron_diff = sig_neuron_mean - bkg_neuron_mean

# Sort by difference
sorted_idx = np.argsort(neuron_diff)[::-1]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top discriminative neurons
ax = axes[0, 0]
top_n = 20
ax.barh(range(top_n), neuron_diff[sorted_idx[:top_n]], color="blue", alpha=0.7)
ax.set_yticks(range(top_n))
ax.set_yticklabels([f"Neuron {i}" for i in sorted_idx[:top_n]])
ax.set_xlabel("Mean Activation Difference (Signal - Background)")
ax.set_title(f"Top {top_n} Neurons Favoring Signal")
ax.invert_yaxis()

ax = axes[0, 1]
ax.barh(range(top_n), neuron_diff[sorted_idx[-top_n:]], color="red", alpha=0.7)
ax.set_yticks(range(top_n))
ax.set_yticklabels([f"Neuron {i}" for i in sorted_idx[-top_n:]])
ax.set_xlabel("Mean Activation Difference (Signal - Background)")
ax.set_title(f"Top {top_n} Neurons Favoring Background")
ax.invert_yaxis()

# Activation heatmap for most discriminative neurons
ax = axes[1, 0]
n_show = min(50, embed_dim)
top_neurons = sorted_idx[: n_show // 2].tolist() + sorted_idx[-n_show // 2 :].tolist()

# Sample jets for heatmap
n_jets_hm = 50
sig_jets = np.where(vis_labels == 1)[0][: n_jets_hm // 2]
bkg_jets = np.where(vis_labels == 0)[0][: n_jets_hm // 2]
jet_order = np.concatenate([sig_jets, bkg_jets])

heatmap_data = final_act[jet_order][:, top_neurons]
im = ax.imshow(
    heatmap_data.T,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-np.percentile(np.abs(heatmap_data), 95),
    vmax=np.percentile(np.abs(heatmap_data), 95),
)
ax.axvline(len(sig_jets) - 0.5, color="black", linestyle="--", linewidth=2)
ax.set_xlabel("Jet Index (Signal | Background)")
ax.set_ylabel("Neuron Index")
ax.set_title("Activation Heatmap (Most Discriminative Neurons)")
plt.colorbar(im, ax=ax, label="Activation")

# Correlation between neuron activations
ax = axes[1, 1]
top_act = final_act[:, sorted_idx[:30]]
corr = np.corrcoef(top_act.T)
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xlabel("Neuron Index")
ax.set_ylabel("Neuron Index")
ax.set_title("Correlation Between Top 30 Discriminative Neurons")
plt.colorbar(im, ax=ax, label="Correlation")

plt.tight_layout()
save_fig(fig, "neuron_activation_patterns")
plt.show()

# ============================================================
# 6. LAYER-WISE SEPARABILITY ANALYSIS
# ============================================================
print("\nComputing layer-wise class separability...")


def compute_separability(activations_arr, labels):
    """Compute Fisher's discriminant ratio as a measure of class separability."""
    sig_act = activations_arr[labels == 1]
    bkg_act = activations_arr[labels == 0]

    sig_mean = sig_act.mean(axis=0)
    bkg_mean = bkg_act.mean(axis=0)

    sig_var = sig_act.var(axis=0) + 1e-8
    bkg_var = bkg_act.var(axis=0) + 1e-8

    # Fisher's discriminant ratio per neuron
    fisher = (sig_mean - bkg_mean) ** 2 / (sig_var + bkg_var)

    return fisher.mean(), fisher.max(), fisher


separability_stats = []
layer_order = (
    ["input_processed", "embedding"]
    + [f"particle_attn_{i}" for i in range(len(activations["particle_attn_layers"]))]
    + ["pre_cls"]
    + [f"cls_attn_{i}" for i in range(len(activations["cls_attn_layers"]))]
    + ["final_cls_token"]
)

layer_acts = [
    activations["input_processed"].mean(dim=1).numpy(),
    activations["embedding"].mean(dim=1).numpy(),
]
for act in activations["particle_attn_layers"]:
    layer_acts.append(act.mean(dim=1).numpy())
layer_acts.append(activations["pre_cls"].mean(dim=1).numpy())
for act in activations["cls_attn_layers"]:
    layer_acts.append(act.mean(dim=1).numpy())
layer_acts.append(activations["final_cls_token"].numpy())

for name, act in zip(layer_order, layer_acts):
    mean_fisher, max_fisher, _ = compute_separability(act, vis_labels)
    separability_stats.append(
        {
            "layer": name,
            "mean_fisher": mean_fisher,
            "max_fisher": max_fisher,
            "dim": act.shape[1],
        }
    )

# Plot separability evolution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x_pos = range(len(separability_stats))
ax.bar(
    x_pos, [s["mean_fisher"] for s in separability_stats], color="steelblue", alpha=0.7
)
ax.set_xticks(x_pos)
ax.set_xticklabels(
    [s["layer"].replace("_", "\n") for s in separability_stats],
    rotation=45,
    ha="right",
    fontsize=8,
)
ax.set_ylabel("Mean Fisher Discriminant Ratio")
ax.set_title("Average Class Separability Across Layers")

ax = axes[1]
ax.bar(
    x_pos, [s["max_fisher"] for s in separability_stats], color="darkorange", alpha=0.7
)
ax.set_xticks(x_pos)
ax.set_xticklabels(
    [s["layer"].replace("_", "\n") for s in separability_stats],
    rotation=45,
    ha="right",
    fontsize=8,
)
ax.set_ylabel("Max Fisher Discriminant Ratio")
ax.set_title("Best Single-Neuron Separability Across Layers")

plt.tight_layout()
save_fig(fig, "layer_separability_evolution")
plt.show()

# ============================================================
# 7. PER-CONSTITUENT ACTIVATION ANALYSIS
# ============================================================
print("\nAnalyzing per-constituent activations...")

# Look at how individual constituents are represented after particle attention
final_part_attn = activations["particle_attn_layers"][-1]  # [B, N, D]

# Compute activation magnitude per constituent
const_act_magnitude = torch.norm(final_part_attn, dim=-1).numpy()  # [B, N]

# Get constituent pT for correlation
const_pt_vis = vis_x[:, :, 1].cpu().numpy()
mask_vis = vis_mask.cpu().numpy()

# Flatten for scatter plot
pt_flat = []
act_flat = []
label_flat = []

for i in range(n_vis):
    n_valid = int(mask_vis[i].sum())
    pt_flat.extend(const_pt_vis[i, :n_valid])
    act_flat.extend(const_act_magnitude[i, :n_valid])
    label_flat.extend([vis_labels[i]] * n_valid)

pt_flat = np.array(pt_flat)
act_flat = np.array(act_flat)
label_flat = np.array(label_flat)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Constituent pT vs activation magnitude
ax = axes[0]
sig_mask_const = label_flat == 1
ax.scatter(
    pt_flat[~sig_mask_const],
    act_flat[~sig_mask_const],
    alpha=0.1,
    s=2,
    c="red",
    label="Background",
)
ax.scatter(
    pt_flat[sig_mask_const],
    act_flat[sig_mask_const],
    alpha=0.1,
    s=2,
    c="blue",
    label="Signal",
)
ax.set_xlabel("Constituent $p_T$ [GeV]")
ax.set_ylabel("Activation Magnitude (L2 norm)")
ax.set_title("Constituent Activation vs $p_T$")
ax.legend()
ax.set_xlim(0, np.percentile(pt_flat, 99))

# Histogram of activation magnitudes
ax = axes[1]
ax.hist(
    act_flat[sig_mask_const],
    bins=50,
    histtype="step",
    label="Signal",
    color="blue",
    density=True,
)
ax.hist(
    act_flat[~sig_mask_const],
    bins=50,
    histtype="step",
    label="Background",
    color="red",
    density=True,
)
ax.set_xlabel("Activation Magnitude")
ax.set_ylabel("Density")
ax.set_title("Per-Constituent Activation Distribution")
ax.legend()

plt.tight_layout()
save_fig(fig, "constituent_activation_analysis")
plt.show()

print(f"\n{'='*60}")
print("Layer activation analysis complete!")
print(f"{'='*60}")

In [ ]:
# ============================================================
# FEATURE IMPORTANCE ANALYSIS
# ============================================================
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F

print("=" * 60)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 60)

# Feature names for analysis
input_feature_names = [
    "Mass",
    "pT",
    "η",
    "φ",
    "d_xy",
    "z_0",
    "Charge",
    "log(pT_rel)",
    "Δη",
    "Δφ",
    "PUPPI weight",
    "log(ΔR)",
]

# Add particle ID features if present
n_features = all_constituents.shape[2]
if n_features > 12:
    for i in range(12, n_features):
        input_feature_names.append(f"ParticleID_{i-11}")

print(f"Analyzing {len(input_feature_names)} input features")

# ============================================================
# 1. GRADIENT-BASED FEATURE IMPORTANCE
# ============================================================
print("\n1. Computing gradient-based feature importance...")

n_grad_samples = min(500, len(all_constituents))
grad_x = all_constituents[:n_grad_samples].clone().to(device)
grad_x.requires_grad_(True)
grad_mask = grad_x[:, :, 1] > 0
grad_labels = all_labels[:n_grad_samples]

model.eval()

# Forward pass
tmep_outputs = model(grad_x, particle_mask=grad_mask)
outputs = tmep_outputs["classification"]

# Compute gradients w.r.t. inputs for both classes
# Gradient for positive class (signal)
grad_x.grad = None
loss_signal = outputs.sum()
loss_signal.backward(retain_graph=True)
grads_signal = grad_x.grad.clone().detach().cpu()

# Reset and compute for sigmoid output
grad_x.grad = None
grad_x.requires_grad_(True)
tmep_outputs = model(grad_x, particle_mask=grad_mask)
outputs = tmep_outputs["classification"]
probs = torch.sigmoid(outputs)
probs.sum().backward()
grads_prob = grad_x.grad.clone().detach().cpu()

# Compute importance as mean absolute gradient per feature
mask_np = grad_mask.cpu().numpy()
grad_importance = np.zeros(len(input_feature_names))
grad_importance_signal = np.zeros(len(input_feature_names))
grad_importance_background = np.zeros(len(input_feature_names))

for feat_idx in range(min(len(input_feature_names), grads_prob.shape[2])):
    # Get gradients for valid constituents only
    valid_grads = []
    sig_grads = []
    bkg_grads = []

    for i in range(n_grad_samples):
        n_valid = int(mask_np[i].sum())
        feat_grads = np.abs(grads_prob[i, :n_valid, feat_idx].numpy())
        valid_grads.extend(feat_grads)

        if grad_labels[i] == 1:
            sig_grads.extend(feat_grads)
        else:
            bkg_grads.extend(feat_grads)

    grad_importance[feat_idx] = np.mean(valid_grads)
    grad_importance_signal[feat_idx] = np.mean(sig_grads) if sig_grads else 0
    grad_importance_background[feat_idx] = np.mean(bkg_grads) if bkg_grads else 0

# Normalize
grad_importance = grad_importance / grad_importance.sum()
grad_importance_signal = grad_importance_signal / grad_importance_signal.sum()
grad_importance_background = (
    grad_importance_background / grad_importance_background.sum()
)

# ============================================================
# 2. PERMUTATION IMPORTANCE
# ============================================================
print("2. Computing permutation importance (this may take a minute)...")

n_perm_samples = min(1000, len(all_constituents))
perm_x = all_constituents[:n_perm_samples].to(device)
perm_mask = perm_x[:, :, 1] > 0
perm_labels = all_labels[:n_perm_samples]

# Get baseline AUC
model.eval()
with torch.no_grad():
    tmep_outputs = model(perm_x, particle_mask=perm_mask)
    baseline_outputs = (
        torch.sigmoid(tmep_outputs["classification"]).squeeze().cpu().numpy()
    )


baseline_auc = roc_auc_score(perm_labels, baseline_outputs)
print(f"  Baseline AUC: {baseline_auc:.4f}")

perm_importance = np.zeros(len(input_feature_names))
n_permutations = 5

for feat_idx in range(min(len(input_feature_names), perm_x.shape[2])):
    auc_drops = []

    for _ in range(n_permutations):
        # Create permuted copy
        perm_x_copy = perm_x.clone()

        # Permute this feature across samples
        perm_indices = torch.randperm(n_perm_samples)
        perm_x_copy[:, :, feat_idx] = perm_x[perm_indices, :, feat_idx]

        # Evaluate
        with torch.no_grad():
            tmep_outputs = model(perm_x_copy, particle_mask=perm_mask)
            perm_outputs = (
                torch.sigmoid(tmep_outputs["classification"]).squeeze().cpu().numpy()
            )

        try:
            perm_auc = roc_auc_score(perm_labels, perm_outputs)
            auc_drops.append(baseline_auc - perm_auc)
        except:
            auc_drops.append(0)

    perm_importance[feat_idx] = np.mean(auc_drops)

    if (feat_idx + 1) % 4 == 0:
        print(
            f"    Processed {feat_idx + 1}/{min(len(input_feature_names), perm_x.shape[2])} features"
        )

# ============================================================
# 3. FEATURE ABLATION (ZEROING OUT)
# ============================================================
print("3. Computing feature ablation importance...")

ablation_importance = np.zeros(len(input_feature_names))

for feat_idx in range(min(len(input_feature_names), perm_x.shape[2])):
    # Create ablated copy
    ablated_x = perm_x.clone()
    ablated_x[:, :, feat_idx] = 0  # Zero out this feature

    # Evaluate
    with torch.no_grad():
        tmep_outputs = model(ablated_x, particle_mask=perm_mask)
        ablated_outputs = (
            torch.sigmoid(tmep_outputs["classification"]).squeeze().cpu().numpy()
        )

    try:
        ablated_auc = roc_auc_score(perm_labels, ablated_outputs)
        ablation_importance[feat_idx] = baseline_auc - ablated_auc
    except:
        ablation_importance[feat_idx] = 0

# ============================================================
# 4. STATISTICAL FEATURE SEPARABILITY
# ============================================================
print("4. Computing statistical feature separability...")

# Use validation dataset (all_constituents is already available)
stat_x = all_constituents[:, :, : len(input_feature_names)].numpy()
stat_mask = (all_constituents[:, :, 1] > 0).numpy()  # pT > 0 means valid
stat_labels = all_labels

# Compute Fisher discriminant ratio for each feature
fisher_scores = np.zeros(len(input_feature_names))
ks_scores = np.zeros(len(input_feature_names))

from scipy.stats import ks_2samp

for feat_idx in range(min(len(input_feature_names), stat_x.shape[2])):
    # Get signal and background values for valid constituents only
    # Extract feature values and mask for signal jets
    sig_feat_vals = stat_x[
        stat_labels == 1, :, feat_idx
    ]  # [n_sig_jets, n_constituents]
    sig_mask_vals = stat_mask[stat_labels == 1]  # [n_sig_jets, n_constituents]
    sig_vals = sig_feat_vals[sig_mask_vals].flatten()  # flatten valid constituents

    # Extract feature values and mask for background jets
    bkg_feat_vals = stat_x[stat_labels == 0, :, feat_idx]
    bkg_mask_vals = stat_mask[stat_labels == 0]
    bkg_vals = bkg_feat_vals[bkg_mask_vals].flatten()

    # Fisher discriminant ratio
    sig_mean, bkg_mean = sig_vals.mean(), bkg_vals.mean()
    sig_var, bkg_var = sig_vals.var() + 1e-8, bkg_vals.var() + 1e-8
    fisher_scores[feat_idx] = (sig_mean - bkg_mean) ** 2 / (sig_var + bkg_var)

    # Kolmogorov-Smirnov test (subsample for speed)
    n_subsample = min(10000, len(sig_vals), len(bkg_vals))
    sig_sample = (
        np.random.choice(sig_vals, n_subsample, replace=False)
        if len(sig_vals) > n_subsample
        else sig_vals
    )
    bkg_sample = (
        np.random.choice(bkg_vals, n_subsample, replace=False)
        if len(bkg_vals) > n_subsample
        else bkg_vals
    )
    ks_stat, _ = ks_2samp(sig_sample, bkg_sample)
    ks_scores[feat_idx] = ks_stat

# ============================================================
# 5. VISUALIZE RESULTS
# ============================================================
print("\nGenerating visualizations...")

# Limit to actual features we have
n_plot_features = min(len(input_feature_names), 12)
plot_feature_names = input_feature_names[:n_plot_features]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Gradient-based importance
ax = axes[0, 0]
sorted_idx = np.argsort(grad_importance[:n_plot_features])[::-1]
y_pos = np.arange(n_plot_features)
ax.barh(
    y_pos, grad_importance[:n_plot_features][sorted_idx], color="steelblue", alpha=0.8
)
ax.set_yticks(y_pos)
ax.set_yticklabels([plot_feature_names[i] for i in sorted_idx])
ax.set_xlabel("Normalized Mean |Gradient|")
ax.set_title("Gradient-Based Importance")
ax.invert_yaxis()

# 2. Permutation importance
ax = axes[0, 1]
sorted_idx = np.argsort(perm_importance[:n_plot_features])[::-1]
colors = [
    "green" if v > 0 else "red" for v in perm_importance[:n_plot_features][sorted_idx]
]
ax.barh(y_pos, perm_importance[:n_plot_features][sorted_idx], color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels([plot_feature_names[i] for i in sorted_idx])
ax.set_xlabel("AUC Drop When Permuted")
ax.set_title("Permutation Importance")
ax.axvline(0, color="black", linestyle="--", alpha=0.5)
ax.invert_yaxis()

# 3. Ablation importance
ax = axes[0, 2]
sorted_idx = np.argsort(ablation_importance[:n_plot_features])[::-1]
colors = [
    "green" if v > 0 else "red"
    for v in ablation_importance[:n_plot_features][sorted_idx]
]
ax.barh(
    y_pos, ablation_importance[:n_plot_features][sorted_idx], color=colors, alpha=0.8
)
ax.set_yticks(y_pos)
ax.set_yticklabels([plot_feature_names[i] for i in sorted_idx])
ax.set_xlabel("AUC Drop When Zeroed")
ax.set_title("Feature Ablation Importance")
ax.axvline(0, color="black", linestyle="--", alpha=0.5)
ax.invert_yaxis()

# 4. Fisher discriminant ratio
ax = axes[1, 0]
sorted_idx = np.argsort(fisher_scores[:n_plot_features])[::-1]
ax.barh(
    y_pos, fisher_scores[:n_plot_features][sorted_idx], color="darkorange", alpha=0.8
)
ax.set_yticks(y_pos)
ax.set_yticklabels([plot_feature_names[i] for i in sorted_idx])
ax.set_xlabel("Fisher Discriminant Ratio")
ax.set_title("Statistical Separability (Fisher)")
ax.invert_yaxis()

# 5. KS statistic
ax = axes[1, 1]
sorted_idx = np.argsort(ks_scores[:n_plot_features])[::-1]
ax.barh(y_pos, ks_scores[:n_plot_features][sorted_idx], color="purple", alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels([plot_feature_names[i] for i in sorted_idx])
ax.set_xlabel("KS Statistic")
ax.set_title("Statistical Separability (KS Test)")
ax.invert_yaxis()

# 6. Combined ranking
ax = axes[1, 2]


# Normalize all scores to [0, 1] and combine
def normalize_scores(scores):
    if scores.max() - scores.min() > 0:
        return (scores - scores.min()) / (scores.max() - scores.min())
    return scores


combined_score = (
    normalize_scores(grad_importance[:n_plot_features])
    + normalize_scores(np.maximum(perm_importance[:n_plot_features], 0))
    + normalize_scores(np.maximum(ablation_importance[:n_plot_features], 0))
    + normalize_scores(fisher_scores[:n_plot_features])
    + normalize_scores(ks_scores[:n_plot_features])
) / 5

sorted_idx = np.argsort(combined_score)[::-1]
ax.barh(y_pos, combined_score[sorted_idx], color="darkgreen", alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels([plot_feature_names[i] for i in sorted_idx])
ax.set_xlabel("Combined Importance Score")
ax.set_title("Overall Feature Ranking")
ax.invert_yaxis()

plt.suptitle(
    "Feature Importance Analysis for Signal vs Background Discrimination", fontsize=14
)
plt.tight_layout()
save_fig(fig, "feature_importance_analysis")
plt.show()

# ============================================================
# 6. SIGNAL VS BACKGROUND GRADIENT COMPARISON
# ============================================================
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(n_plot_features)
width = 0.35

ax.bar(
    x - width / 2,
    grad_importance_signal[:n_plot_features],
    width,
    label="Signal (b-jets)",
    color="blue",
    alpha=0.7,
)
ax.bar(
    x + width / 2,
    grad_importance_background[:n_plot_features],
    width,
    label="Background",
    color="red",
    alpha=0.7,
)

ax.set_xlabel("Feature")
ax.set_ylabel("Normalized Mean |Gradient|")
ax.set_title("Gradient Importance: Signal vs Background")
ax.set_xticks(x)
ax.set_xticklabels(plot_feature_names, rotation=45, ha="right")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
save_fig(fig, "gradient_importance_by_class")
plt.show()

# ============================================================
# 7. FEATURE CORRELATION WITH MODEL OUTPUT
# ============================================================
print("\nComputing feature correlations with model output...")

# Get model predictions
with torch.no_grad():
    tmep_outputs = model(perm_x_copy, particle_mask=perm_mask)
    pred_outputs = torch.sigmoid(tmep_outputs["classification"]).squeeze().cpu().numpy()

# Compute mean feature values per jet (for valid constituents)
mean_features = np.zeros((n_perm_samples, n_plot_features))
for i in range(n_perm_samples):
    n_valid = int(perm_mask[i].sum().item())
    for f in range(n_plot_features):
        mean_features[i, f] = perm_x[i, :n_valid, f].cpu().numpy().mean()

# Correlation with predictions
feature_pred_corr = np.zeros(n_plot_features)
for f in range(n_plot_features):
    feature_pred_corr[f] = np.corrcoef(mean_features[:, f], pred_outputs)[0, 1]

fig, ax = plt.subplots(figsize=(12, 6))
colors = ["blue" if c > 0 else "red" for c in feature_pred_corr]
bars = ax.bar(range(n_plot_features), feature_pred_corr, color=colors, alpha=0.7)
ax.set_xticks(range(n_plot_features))
ax.set_xticklabels(plot_feature_names, rotation=45, ha="right")
ax.set_ylabel("Pearson Correlation")
ax.set_xlabel("Feature")
ax.set_title("Feature Correlation with Model Output (b-tag Score)")
ax.axhline(0, color="black", linestyle="--", alpha=0.5)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
save_fig(fig, "feature_correlation_with_output")
plt.show()

# ============================================================
# 8. SUMMARY TABLE
# ============================================================
print("\n" + "=" * 80)
print("FEATURE IMPORTANCE SUMMARY")
print("=" * 80)
print(
    f"{'Feature':<15} {'Gradient':>10} {'Permute':>10} {'Ablation':>10} {'Fisher':>10} {'KS':>10} {'Combined':>10}"
)
print("-" * 80)

for i in range(n_plot_features):
    print(
        f"{plot_feature_names[i]:<15} "
        f"{grad_importance[i]:>10.4f} "
        f"{perm_importance[i]:>10.4f} "
        f"{ablation_importance[i]:>10.4f} "
        f"{fisher_scores[i]:>10.4f} "
        f"{ks_scores[i]:>10.4f} "
        f"{combined_score[i]:>10.4f}"
    )

# Top features summary
print("\n" + "=" * 80)
print("TOP 5 MOST IMPORTANT FEATURES (by combined score):")
print("=" * 80)
top_5_idx = np.argsort(combined_score)[::-1][:5]
for rank, idx in enumerate(top_5_idx, 1):
    print(f"  {rank}. {plot_feature_names[idx]} (score: {combined_score[idx]:.4f})")

print(f"\n{'='*60}")
print("Feature importance analysis complete!")
print(f"{'='*60}")

In [ ]:
# ============================================================
# MODEL BEHAVIOR ANALYSIS & MAXIMUM DISCRIMINATIVE POWER
# ============================================================
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss
from scipy.stats import entropy
import warnings

warnings.filterwarnings("ignore")

print("=" * 70)
print("MODEL BEHAVIOR ANALYSIS & MAXIMUM DISCRIMINATIVE POWER ESTIMATION")
print("=" * 70)

# ============================================================
# 1. ERROR ANALYSIS - UNDERSTANDING MISCLASSIFICATIONS
# ============================================================
print("\n1. ERROR ANALYSIS")
print("-" * 50)

# Get predictions at different thresholds
threshold = 0.5
# cls_outputs = torch.sigmoid(all_outputs["classification"]).squeeze().cpu().numpy()
predictions = (all_outputs > threshold).astype(int)
correct = predictions == all_labels
incorrect = ~correct

all_labels = all_labels.squeeze()

# Classify error types
false_positives = (predictions == 1) & (all_labels == 0)
false_negatives = (predictions == 0) & (all_labels == 1)
true_positives = (predictions == 1) & (all_labels == 1)
true_negatives = (predictions == 0) & (all_labels == 0)

print(f"Threshold: {threshold}")
print(
    f"  True Positives:  {true_positives.sum():>6} ({100*true_positives.sum()/all_labels.sum():.1f}% of signal)"
)
print(
    f"  True Negatives:  {true_negatives.sum():>6} ({100*true_negatives.sum()/(len(all_labels)-all_labels.sum()):.1f}% of background)"
)
print(
    f"  False Positives: {false_positives.sum():>6} (background misclassified as signal)"
)
print(
    f"  False Negatives: {false_negatives.sum():>6} (signal misclassified as background)"
)

# Analyze characteristics of misclassified samples
print("\n  Kinematic properties of misclassified jets:")

# Jet pT for different categories
fp_pt = val_jet_pt[false_positives]
fn_pt = val_jet_pt[false_negatives]
tp_pt = val_jet_pt[true_positives]
tn_pt = val_jet_pt[true_negatives]

print(
    f"    False Positives: mean pT = {fp_pt.mean():.1f} GeV, mean |η| = {np.abs(val_jet_eta[false_positives]).mean():.2f}"
)
print(
    f"    False Negatives: mean pT = {fn_pt.mean():.1f} GeV, mean |η| = {np.abs(val_jet_eta[false_negatives]).mean():.2f}"
)
print(
    f"    True Positives:  mean pT = {tp_pt.mean():.1f} GeV, mean |η| = {np.abs(val_jet_eta[true_positives]).mean():.2f}"
)
print(
    f"    True Negatives:  mean pT = {tn_pt.mean():.1f} GeV, mean |η| = {np.abs(val_jet_eta[true_negatives]).mean():.2f}"
)

# ============================================================
# 2. CONFIDENCE CALIBRATION
# ============================================================
print("\n2. CONFIDENCE CALIBRATION")
print("-" * 50)

# Compute calibration metrics
brier = brier_score_loss(all_labels, all_outputs)
logloss = log_loss(all_labels, np.clip(all_outputs, 1e-7, 1 - 1e-7))

print(f"  Brier Score: {brier:.4f} (lower is better, perfect = 0)")
print(f"  Log Loss: {logloss:.4f} (lower is better)")

# Calibration curve
prob_true, prob_pred = calibration_curve(
    all_labels, all_outputs, n_bins=10, strategy="uniform"
)

print(f"\n  Calibration by probability bin:")
print(f"  {'Predicted':>12} {'Actual':>12} {'Difference':>12}")
for pt, pp in zip(prob_true, prob_pred):
    print(f"  {pp:>12.3f} {pt:>12.3f} {pt-pp:>+12.3f}")

# Expected Calibration Error (ECE)
bin_edges = np.linspace(0, 1, 11)
ece = 0
for i in range(10):
    mask = (all_outputs >= bin_edges[i]) & (all_outputs < bin_edges[i + 1])
    if mask.sum() > 0:
        bin_acc = all_labels[mask].mean()
        bin_conf = all_outputs[mask].mean()
        ece += mask.sum() * np.abs(bin_acc - bin_conf)
ece /= len(all_labels)
print(f"\n  Expected Calibration Error (ECE): {ece:.4f}")

# ============================================================
# 3. HARD SAMPLE ANALYSIS
# ============================================================
print("\n3. HARD SAMPLE ANALYSIS")
print("-" * 50)

# Define "hard" samples as those with high uncertainty (close to 0.5)
uncertainty = np.abs(all_outputs - 0.5)
hard_threshold = 0.2  # samples with output between 0.3 and 0.7
hard_samples = uncertainty < hard_threshold
easy_samples = uncertainty >= 0.4

print(
    f"  Hard samples (output in [0.3, 0.7]): {hard_samples.sum()} ({100*hard_samples.mean():.1f}%)"
)
print(
    f"  Easy samples (output < 0.1 or > 0.9): {easy_samples.sum()} ({100*easy_samples.mean():.1f}%)"
)

# Accuracy on easy vs hard samples
hard_acc = (predictions[hard_samples] == all_labels[hard_samples]).mean()
easy_acc = (predictions[easy_samples] == all_labels[easy_samples]).mean()
print(f"\n  Accuracy on hard samples: {100*hard_acc:.1f}%")
print(f"  Accuracy on easy samples: {100*easy_acc:.1f}%")

# What makes samples hard?
print("\n  Characteristics of hard samples:")
hard_mask = all_constituents[:, :, 1] > 0
hard_n_const = hard_mask[hard_samples].sum(dim=1).float().mean().item()
easy_n_const = hard_mask[easy_samples].sum(dim=1).float().mean().item()
print(f"    Mean constituents (hard): {hard_n_const:.1f}")
print(f"    Mean constituents (easy): {easy_n_const:.1f}")
print(f"    Mean pT (hard): {val_jet_pt[hard_samples].mean():.1f} GeV")
print(f"    Mean pT (easy): {val_jet_pt[easy_samples].mean():.1f} GeV")

# ============================================================
# 4. MAXIMUM DISCRIMINATIVE POWER ESTIMATION
# ============================================================
print("\n4. MAXIMUM DISCRIMINATIVE POWER ESTIMATION")
print("-" * 50)

# Method 1: Class overlap estimation using k-NN
print("\n  Method 1: k-NN Class Overlap Estimation")

# Use a subset for speed
n_knn = min(50000, len(all_constituents))
knn_x = all_constituents[:n_knn]
knn_mask = knn_x[:, :, 1] > 0

# Compute mean features per jet for k-NN
knn_features = []
for i in range(n_knn):
    n_valid = int(knn_mask[i].sum().item())
    jet_feats = knn_x[i, :n_valid, :12].mean(dim=0).numpy()  # Mean over constituents
    knn_features.append(jet_feats)
knn_features = np.array(knn_features)
knn_labels = all_labels[:n_knn]

# Fit k-NN classifier with different k values
knn_aucs = []
for k in [1, 3, 5, 11, 21]:
    knn = KNeighborsClassifier(n_neighbors=k, weights="distance")
    # Use cross-validation-like split
    train_idx = np.random.choice(n_knn, int(0.7 * n_knn), replace=False)
    test_idx = np.setdiff1d(np.arange(n_knn), train_idx)

    knn.fit(knn_features[train_idx], knn_labels[train_idx])
    knn_proba = knn.predict_proba(knn_features[test_idx])[:, 1]
    knn_auc = roc_auc_score(knn_labels[test_idx], knn_proba)
    knn_aucs.append(knn_auc)
    print(f"    k={k:>2}: AUC = {knn_auc:.4f}")

best_knn_auc = max(knn_aucs)
print(f"    Best k-NN AUC (upper bound estimate): {best_knn_auc:.4f}")

# Method 2: Bayes error estimation via density overlap
print("\n  Method 2: Feature Distribution Overlap Analysis")

# Compute overlap for each feature
from scipy.stats import gaussian_kde

feature_overlaps = []
for feat_idx in range(12):
    sig_vals = knn_features[knn_labels == 1, feat_idx]
    bkg_vals = knn_features[knn_labels == 0, feat_idx]

    # Subsample for speed
    n_sub = min(1000, len(sig_vals), len(bkg_vals))
    sig_sub = np.random.choice(sig_vals, n_sub, replace=False)
    bkg_sub = np.random.choice(bkg_vals, n_sub, replace=False)

    try:
        # Compute KDE overlap
        x_range = np.linspace(
            min(sig_sub.min(), bkg_sub.min()), max(sig_sub.max(), bkg_sub.max()), 100
        )
        sig_kde = gaussian_kde(sig_sub)
        bkg_kde = gaussian_kde(bkg_sub)

        sig_pdf = sig_kde(x_range)
        bkg_pdf = bkg_kde(x_range)

        # Overlap coefficient (Bhattacharyya)
        overlap = np.sum(np.sqrt(sig_pdf * bkg_pdf)) * (x_range[1] - x_range[0])
        feature_overlaps.append(overlap)
    except:
        feature_overlaps.append(1.0)

feature_overlaps = np.array(feature_overlaps)
mean_overlap = feature_overlaps.mean()

# Lower overlap = more separable
estimated_max_auc = 1 - mean_overlap / 2  # Rough approximation
print(f"    Mean feature overlap: {mean_overlap:.4f}")
print(f"    Estimated max AUC (from overlap): {estimated_max_auc:.4f}")

# Method 3: Theoretical bounds from class distributions
print("\n  Method 3: Theoretical Bounds")

# Class balance
n_signal = all_labels.sum()
n_background = len(all_labels) - n_signal
class_ratio = n_signal / len(all_labels)
print(
    f"    Class balance: {100*class_ratio:.1f}% signal / {100*(1-class_ratio):.1f}% background"
)

# Random classifier AUC
print(f"    Random classifier AUC: 0.5000")
print(f"    Current model AUC: {roc_auc_score(all_labels, all_outputs):.4f}")
print(f"    Best k-NN AUC: {best_knn_auc:.4f}")

# Gap analysis
current_auc = roc_auc_score(all_labels, all_outputs)
gap_to_knn = best_knn_auc - current_auc
gap_to_perfect = 1.0 - current_auc

print(f"\n    Performance gaps:")
print(f"      Gap to k-NN (achievable): {gap_to_knn:+.4f}")
print(f"      Gap to perfect: {gap_to_perfect:+.4f}")

# ============================================================
# 5. IMPROVEMENT RECOMMENDATIONS
# ============================================================
print("\n5. IMPROVEMENT RECOMMENDATIONS")
print("-" * 50)

recommendations = []

# Check calibration
if ece > 0.05:
    recommendations.append(
        "• Model is poorly calibrated (ECE > 0.05). Consider temperature scaling or Platt calibration."
    )

# Check class balance impact
if class_ratio < 0.3 or class_ratio > 0.7:
    recommendations.append(
        "• Class imbalance detected. Consider focal loss or class weighting."
    )

# Check for overfitting potential
if best_knn_auc < current_auc:
    recommendations.append(
        "• Model outperforms k-NN, suggesting good feature learning."
    )
else:
    recommendations.append(
        f"• k-NN achieves higher AUC ({best_knn_auc:.4f} vs {current_auc:.4f}). Model may benefit from more capacity or training."
    )

# Check hard samples
if hard_samples.mean() > 0.3:
    recommendations.append(
        f"• {100*hard_samples.mean():.0f}% of samples are hard. Consider curriculum learning or sample weighting."
    )

# Check kinematic dependencies
pt_corr = np.corrcoef(val_jet_pt, all_outputs)[0, 1]
if abs(pt_corr) > 0.3:
    recommendations.append(
        f"• Strong pT-score correlation ({pt_corr:.2f}). Consider pT reweighting for robustness."
    )

for rec in recommendations:
    print(f"  {rec}")

if not recommendations:
    print(
        "  • Model appears well-optimized! Consider ensemble methods or architectural changes for further improvement."
    )

# ============================================================
# 6. VISUALIZATION
# ============================================================
print("\nGenerating diagnostic plots...")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Calibration plot
ax = axes[0, 0]
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.plot(prob_pred, prob_true, "bo-", label="Model")
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives")
ax.set_title("Calibration Curve")
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Error analysis by pT
ax = axes[0, 1]
pt_bins = np.linspace(0, 300, 20)
fp_hist, _ = np.histogram(fp_pt, bins=pt_bins, density=True)
fn_hist, _ = np.histogram(fn_pt, bins=pt_bins, density=True)
tp_hist, _ = np.histogram(tp_pt, bins=pt_bins, density=True)
tn_hist, _ = np.histogram(tn_pt, bins=pt_bins, density=True)

pt_centers = (pt_bins[:-1] + pt_bins[1:]) / 2
ax.plot(pt_centers, fp_hist, "r-", label="False Positives", alpha=0.8)
ax.plot(pt_centers, fn_hist, "b-", label="False Negatives", alpha=0.8)
ax.set_xlabel("Jet pT [GeV]")
ax.set_ylabel("Density")
ax.set_title("pT Distribution of Misclassified Jets")
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Confidence distribution
ax = axes[0, 2]
ax.hist(
    all_outputs[all_labels == 1],
    bins=50,
    range=(0, 1),
    histtype="step",
    label="Signal",
    color="blue",
    density=True,
)
ax.hist(
    all_outputs[all_labels == 0],
    bins=50,
    range=(0, 1),
    histtype="step",
    label="Background",
    color="red",
    density=True,
)
ax.axvline(0.5, color="black", linestyle="--", alpha=0.5)
ax.axvspan(0.3, 0.7, alpha=0.1, color="gray", label="Hard region")
ax.set_xlabel("Model Output")
ax.set_ylabel("Density")
ax.set_title("Output Distribution with Hard Region")
ax.legend()

# 4. AUC vs pT
ax = axes[1, 0]
pt_bin_edges = [25, 50, 75, 100, 150, 200, 300, 500]
pt_aucs = []
pt_counts = []
for i in range(len(pt_bin_edges) - 1):
    mask = (val_jet_pt >= pt_bin_edges[i]) & (val_jet_pt < pt_bin_edges[i + 1])
    if mask.sum() > 50 and len(np.unique(all_labels[mask])) > 1:
        pt_aucs.append(roc_auc_score(all_labels[mask], all_outputs[mask]))
        pt_counts.append(mask.sum())
    else:
        pt_aucs.append(np.nan)
        pt_counts.append(mask.sum())

pt_centers = [
    (pt_bin_edges[i] + pt_bin_edges[i + 1]) / 2 for i in range(len(pt_bin_edges) - 1)
]
ax.bar(range(len(pt_aucs)), pt_aucs, color="steelblue", alpha=0.7)
ax.set_xticks(range(len(pt_aucs)))
ax.set_xticklabels(
    [f"{pt_bin_edges[i]}-{pt_bin_edges[i+1]}" for i in range(len(pt_bin_edges) - 1)],
    rotation=45,
)
ax.set_xlabel("pT Range [GeV]")
ax.set_ylabel("AUC")
ax.set_title("AUC vs Jet pT")
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.3, axis="y")

# 5. Feature overlap visualization
ax = axes[1, 1]
feature_names_short = [
    "Mass",
    "pT",
    "η",
    "φ",
    "d_xy",
    "z_0",
    "Q",
    "log(pT_rel)",
    "Δη",
    "Δφ",
    "PUPPI",
    "log(ΔR)",
]
sorted_idx = np.argsort(feature_overlaps)
ax.barh(
    range(len(feature_overlaps)),
    feature_overlaps[sorted_idx],
    color="darkorange",
    alpha=0.7,
)
ax.set_yticks(range(len(feature_overlaps)))
ax.set_yticklabels([feature_names_short[i] for i in sorted_idx])
ax.set_xlabel("Distribution Overlap (lower = more separable)")
ax.set_title("Feature Separability")
ax.invert_yaxis()

# 6. Summary metrics
ax = axes[1, 2]
ax.axis("off")
summary_text = f"""
MODEL PERFORMANCE SUMMARY
{'='*40}

Current Performance:
  • AUC: {current_auc:.4f}
  • Brier Score: {brier:.4f}
  • Log Loss: {logloss:.4f}
  • ECE: {ece:.4f}

Error Analysis:
  • False Positive Rate: {100*false_positives.sum()/(len(all_labels)-all_labels.sum()):.2f}%
  • False Negative Rate: {100*false_negatives.sum()/all_labels.sum():.2f}%
  • Hard Samples: {100*hard_samples.mean():.1f}%

Maximum Performance Estimate:
  • k-NN Upper Bound: {best_knn_auc:.4f}
  • From Feature Overlap: {estimated_max_auc:.4f}
  • Improvement Potential: {max(0, best_knn_auc - current_auc):.4f}

Data Characteristics:
  • Samples: {len(all_labels):,}
  • Signal: {int(n_signal):,} ({100*class_ratio:.1f}%)
  • Background: {int(n_background):,} ({100*(1-class_ratio):.1f}%)
"""
ax.text(
    0.05,
    0.95,
    summary_text,
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment="top",
    fontfamily="monospace",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
)

plt.suptitle("Model Behavior Analysis & Maximum Discriminative Power", fontsize=14)
plt.tight_layout()
save_fig(fig, "model_behavior_analysis")
plt.show()

print(f"\n{'='*70}")
print("Model behavior analysis complete!")
print(f"{'='*70}")

In [ ]:
# Verify that feature distributions are preserved after stratified split
import matplotlib.pyplot as plt
import vector

import importlib
import train_part_data

importlib.reload(train_part_data)
from train_part_data import CombinedJetDataLoader


def verify_distribution_preservation(
    combined_loader: CombinedJetDataLoader,
    n_bins: int = 50,
    figsize: Tuple[int, int] = (15, 10),
):
    """
    Verify that the stratified split preserves feature distributions
    by comparing original vs train vs validation distributions.
    """

    fig, axes = plt.subplots(2, 4, figsize=figsize)

    for row, (name, orig_ds, train_ds, val_ds) in enumerate(
        [
            (
                "PF",
                combined_loader.pf_dataset,
                combined_loader.train_pf,
                combined_loader.val_pf,
            ),
            (
                "PUPPI",
                combined_loader.puppi_dataset,
                combined_loader.train_puppi,
                combined_loader.val_puppi,
            ),
        ]
    ):
        # Get indices
        train_indices = np.array(train_ds.indices)
        val_indices = np.array(val_ds.indices)

        # Get constituent data
        X_orig = orig_ds.X.numpy()
        X_train = X_orig[train_indices]
        X_val = X_orig[val_indices]

        # Feature names (mass, pt, eta, phi)
        feature_names = ["Mass", "pT", "η", "φ"]

        for col, (feat_idx, feat_name) in enumerate(zip(range(4), feature_names)):
            ax = axes[row, col]

            # Get non-masked values for each split
            mask_orig = orig_ds.mask.numpy()
            mask_train = mask_orig[train_indices]
            mask_val = mask_orig[val_indices]

            vals_orig = X_orig[:, :, feat_idx][mask_orig].flatten()
            vals_train = X_train[:, :, feat_idx][mask_train].flatten()
            vals_val = X_val[:, :, feat_idx][mask_val].flatten()

            # Compute appropriate bins
            if feat_name == "pT":
                bins = np.linspace(0, np.percentile(vals_orig, 99), n_bins)
            elif feat_name == "Mass":
                bins = np.linspace(0, np.percentile(vals_orig, 99), n_bins)
            else:
                bins = np.linspace(vals_orig.min(), vals_orig.max(), n_bins)

            # Plot normalized histograms
            ax.hist(
                vals_orig,
                bins=bins,
                histtype="step",
                density=True,
                label="Original",
                color="black",
                linewidth=2,
            )
            ax.hist(
                vals_train,
                bins=bins,
                histtype="step",
                density=True,
                label="Train",
                color="blue",
                linewidth=1.5,
                linestyle="--",
            )
            ax.hist(
                vals_val,
                bins=bins,
                histtype="step",
                density=True,
                label="Val",
                color="red",
                linewidth=1.5,
                linestyle=":",
            )

            ax.set_xlabel(feat_name)
            ax.set_ylabel("Density")
            ax.set_title(f"{name} - {feat_name}")
            if row == 0 and col == 0:
                ax.legend()

    plt.tight_layout()
    plt.suptitle("Stratified Split: Feature Distribution Preservation", y=1.02)
    plt.show()


combined_loader_mode2 = CombinedJetDataLoader(
    pf_data_path=config_part["training"]["data"]["pf_data_path"],
    puppi_data_path=config_part["training"]["data"]["puppi_data_path"],
    val_split=config_part["training"]["data"]["val_split"],
    batch_size=config_part["training"]["batch_size"],
    match_mode="size_and_ratio",
    num_workers=config_part["training"]["data"]["num_workers"],
    random_state=42,
)
# Run the verification
verify_distribution_preservation(combined_loader_mode2)

In [ ]:
plt.hist(jet_pt, bins=50, range=(0, 500), histtype="step", density=True)

train_loader_pf_comb, train_loader_puppi_comb = (
    combined_loader_mode2.get_train_loaders()
)
val_loader_pf_comb, val_loader_puppi_comb = combined_loader_mode2.get_val_loaders()
all_constituents_pf_comb_train = []
all_constituents_pf_comb_val = []
for batch in train_loader_pf_comb:
    all_constituents_pf_comb_train.append(batch[0].numpy())
for batch in val_loader_pf_comb:
    all_constituents_pf_comb_val.append(batch[0].numpy())
all_constituents_pf_comb_train = np.concatenate(all_constituents_pf_comb_train, axis=0)
all_constituents_pf_comb_val = np.concatenate(all_constituents_pf_comb_val, axis=0)

const_pf_comb_train_4_vec = vector.array(
    {
        "pt": all_constituents_pf_comb_train[:, :, 1],
        "eta": all_constituents_pf_comb_train[:, :, 2],
        "phi": all_constituents_pf_comb_train[:, :, 3],
        "mass": all_constituents_pf_comb_train[:, :, 0],
    }
)
const_pf_comb_val_4_vec = vector.array(
    {
        "pt": all_constituents_pf_comb_val[:, :, 1],
        "eta": all_constituents_pf_comb_val[:, :, 2],
        "phi": all_constituents_pf_comb_val[:, :, 3],
        "mass": all_constituents_pf_comb_val[:, :, 0],
    }
)
jet_pf_comb_train_4_vec = const_pf_comb_train_4_vec.sum(axis=1)
jet_pf_comb_val_4_vec = const_pf_comb_val_4_vec.sum(axis=1)
plt.hist(
    jet_pt,
    bins=50,
    range=(0, 500),
    histtype="step",
    density=True,
    label="Full dataset val",
)
plt.hist(
    jet_vectors_train.pt,
    bins=50,
    range=(0, 500),
    histtype="step",
    density=True,
    label="Full dataset Train",
)
plt.hist(
    jet_pf_comb_train_4_vec.pt,
    bins=50,
    range=(0, 500),
    histtype="step",
    density=True,
    label="Downsampled Train",
)
plt.hist(
    jet_pf_comb_val_4_vec.pt,
    bins=50,
    range=(0, 500),
    histtype="step",
    density=True,
    label="Downsampled Val",
)
plt.legend()
plt.show()